# Fabric Notebook Scanner v2

Combines two scanners:

- **Tenant-scale enumeration** (v1) — async fanout via Fabric admin/user APIs
  *or* in-place Lakehouse Files scanning. Cross-workspace tracking,
  workspace-name resolution, runtime-built write detection, review queue.
- **NotebookAudit engine** (slavatrofimov) — 100+ categorised regex patterns
  for exfiltration / cloud / webhook / paste-service / credential / shell /
  Spark / streaming patterns, AST-based suspicious-import + eval/exec
  detection, Shannon-entropy obfuscation detection, and a multi-format code
  extractor that handles Fabric `definition.parts` base64 envelopes.

## Output
One Delta table (`notebook_url_scan`) with one row per finding:
- `category` / `severity` / `message` — NotebookAudit-style triage
- `direction` / `cross_workspace` / `dest_workspace_*` — v1 data-flow tracking
- `cell_index` / `line_number` / `code_snippet` — precise locations
- Credential snippets are auto-redacted before persist.


In [ ]:
# === Configuration ===========================================================

# -----------------------------------------------------------------------------
# READ FROM — which workspaces & notebooks to scan
# -----------------------------------------------------------------------------
SOURCE_MODE = "api"          # "api" or "lakehouse"

# --- API-mode config ---------------------------------------------------------
ADMIN_MODE = True
READ_WORKSPACE_IDS = None    # None or ["guid1", ...]
MAX_NOTEBOOKS = None         # int for smoke test, None for full

# --- Lakehouse-mode config ---------------------------------------------------
# Leave the four SOURCE_LAKEHOUSE_* values blank to auto-detect the lakehouse
# that's currently mounted as the notebook's default. Cell 2 will resolve the
# missing pieces via notebookutils.runtime.context and print what it found.
# Fill them in explicitly only when you want to scan a lakehouse other than
# the one mounted to this notebook.
SOURCE_LAKEHOUSE_WORKSPACE_ID   = ""
SOURCE_LAKEHOUSE_WORKSPACE_NAME = ""
SOURCE_LAKEHOUSE_ID             = ""
SOURCE_LAKEHOUSE_NAME           = ""
SOURCE_SUBPATH                  = "Files/notebooks"
SOURCE_FILE_GLOB                = "*.ipynb"

# -----------------------------------------------------------------------------
# WRITE TO — where findings land
# -----------------------------------------------------------------------------
WRITE_TO_DEFAULT_LAKEHOUSE = True
WRITE_WORKSPACE_ID  = "<output-workspace-guid>"
WRITE_LAKEHOUSE_ID  = "<output-lakehouse-guid>"
WRITE_LAKEHOUSE_NAME = "<output-lakehouse-name>"
WRITE_SCHEMA = None

INVENTORY_TABLE = "notebook_inventory"
OUTPUT_TABLE    = "notebook_url_scan"

# -----------------------------------------------------------------------------
# Scanning behaviour
# -----------------------------------------------------------------------------
# Minimum severity to persist. Lower-severity findings still get detected but
# are filtered before insert. Values: "low", "medium", "high", "critical".
MIN_SEVERITY = "low"

# Whether to also scan markdown cells and notebook outputs for URLs.
# (Code cells always get the full pattern + AST + entropy battery.)
SCAN_MARKDOWN_AND_OUTPUTS = True

# Max bytes of code per finding's snippet (auto-redacted for credentials).
MAX_SNIPPET_BYTES = 200

# -----------------------------------------------------------------------------
# Tuning
# -----------------------------------------------------------------------------
TARGET_PARTITION_SIZE = 200
EXECUTOR_CONCURRENCY  = 60
MAX_RETRIES           = 6
TOKEN_AUDIENCE        = "pbi"

FABRIC_BASE = "https://api.fabric.microsoft.com"
PBI_BASE    = "https://api.powerbi.com"


In [ ]:
# === Resolve read/write targets ==============================================
ONELAKE_HOST = "onelake.dfs.fabric.microsoft.com"


def _table_path(workspace_id, lakehouse_id, table, schema=None):
    if schema:
        return (f"abfss://{workspace_id}@{ONELAKE_HOST}/"
                f"{lakehouse_id}/Tables/{schema}/{table}")
    return (f"abfss://{workspace_id}@{ONELAKE_HOST}/"
            f"{lakehouse_id}/Tables/{table}")


def _files_path(workspace_id, lakehouse_id, subpath):
    sub = (subpath or "").lstrip("/")
    return (f"abfss://{workspace_id}@{ONELAKE_HOST}/"
            f"{lakehouse_id}/{sub}" if sub
            else f"abfss://{workspace_id}@{ONELAKE_HOST}/{lakehouse_id}")


def _detect_notebook_runtime():
    """Inspect notebookutils.runtime.context (or mssparkutils) to figure out
    which workspace + lakehouse this notebook is currently attached to.
    Returns a dict; missing keys are empty strings. Empty dict on failure
    (e.g., running outside Fabric, or no default lakehouse mounted)."""
    try:
        import notebookutils as _nbu  # noqa
    except ImportError:
        try:
            import mssparkutils as _nbu  # noqa
        except ImportError:
            return {}
    try:
        ctx = _nbu.runtime.context
    except Exception:
        return {}

    def _g(*keys):
        for k in keys:
            try:
                v = ctx.get(k) if hasattr(ctx, "get") else getattr(ctx, k, None)
            except Exception:
                v = None
            if v:
                return v
        return ""

    return {
        "current_workspace_id":   _g("currentWorkspaceId", "workspaceId"),
        "current_workspace_name": _g("currentWorkspaceName", "workspaceName"),
        "default_lakehouse_id":   _g("defaultLakehouseId", "defaultLakehouse"),
        "default_lakehouse_name": _g("defaultLakehouseName"),
        "default_lakehouse_workspace_id":
            _g("defaultLakehouseWorkspaceId"),
        "default_lakehouse_workspace_name":
            _g("defaultLakehouseWorkspaceName"),
    }


# Capture the detected runtime context once. Other cells read from this dict
# (it's broadcast to executors via the source-LH constants below, and the
# diagnostics cell prints it back to the user).
_NB_RUNTIME = _detect_notebook_runtime() if SOURCE_MODE == "lakehouse" else {}

# Auto-fill any SOURCE_LAKEHOUSE_* value the user left blank from the detected
# runtime context. Explicit user-set values always win.
if SOURCE_MODE == "lakehouse" and _NB_RUNTIME:
    if not SOURCE_LAKEHOUSE_ID:
        SOURCE_LAKEHOUSE_ID = _NB_RUNTIME.get("default_lakehouse_id", "") or ""
    if not SOURCE_LAKEHOUSE_NAME:
        SOURCE_LAKEHOUSE_NAME = (_NB_RUNTIME.get("default_lakehouse_name", "")
                                 or "")
    if not SOURCE_LAKEHOUSE_WORKSPACE_ID:
        SOURCE_LAKEHOUSE_WORKSPACE_ID = (
            _NB_RUNTIME.get("default_lakehouse_workspace_id", "")
            or _NB_RUNTIME.get("current_workspace_id", "") or "")
    if not SOURCE_LAKEHOUSE_WORKSPACE_NAME:
        SOURCE_LAKEHOUSE_WORKSPACE_NAME = (
            _NB_RUNTIME.get("default_lakehouse_workspace_name", "")
            or _NB_RUNTIME.get("current_workspace_name", "") or "")


if WRITE_TO_DEFAULT_LAKEHOUSE:
    inventory_target = INVENTORY_TABLE
    output_target    = OUTPUT_TABLE
    write_kind = "saveAsTable (default attached Lakehouse)"
else:
    inventory_target = _table_path(WRITE_WORKSPACE_ID, WRITE_LAKEHOUSE_ID,
                                   INVENTORY_TABLE, WRITE_SCHEMA)
    output_target    = _table_path(WRITE_WORKSPACE_ID, WRITE_LAKEHOUSE_ID,
                                   OUTPUT_TABLE, WRITE_SCHEMA)
    write_kind = f"ABFSS Delta path (workspace {WRITE_WORKSPACE_ID})"

if SOURCE_MODE == "lakehouse":
    if SOURCE_LAKEHOUSE_WORKSPACE_ID and SOURCE_LAKEHOUSE_ID:
        source_uri = _files_path(SOURCE_LAKEHOUSE_WORKSPACE_ID,
                                 SOURCE_LAKEHOUSE_ID, SOURCE_SUBPATH)
        if _NB_RUNTIME and SOURCE_LAKEHOUSE_ID == _NB_RUNTIME.get(
                "default_lakehouse_id"):
            source_kind_label = (f"mounted lakehouse "
                                 f"(auto-detected from notebookutils context)")
        else:
            source_kind_label = (f"ABFSS Lakehouse path "
                                 f"(workspace {SOURCE_LAKEHOUSE_WORKSPACE_ID})")
    else:
        # No workspace+lakehouse pair available — fall back to a relative path
        # that Spark resolves against the notebook's default attached LH.
        # The output rows will lose the workspace_id / source_lakehouse_id
        # provenance (no way to know which LH "Files/..." resolves to without
        # a runtime context that we couldn't get).
        source_uri = (SOURCE_SUBPATH or "Files")
        source_kind_label = ("attached default Lakehouse (relative — could not "
                             "detect mounted LH; workspace/source_lakehouse "
                             "columns will be empty)")
else:
    source_uri = None
    source_kind_label = (("admin scanner (all workspaces)" if ADMIN_MODE
                          else "user-accessible workspaces"))

print("============================================================")
print(" Fabric Notebook Scanner v2 — runtime configuration")
print("============================================================")
print(f" SOURCE MODE      : {SOURCE_MODE}")
print(f" READ FROM        : {source_kind_label}")
if SOURCE_MODE == "lakehouse":
    if _NB_RUNTIME:
        ws_n = _NB_RUNTIME.get("current_workspace_name", "")
        ws_i = _NB_RUNTIME.get("current_workspace_id", "")
        lh_n = _NB_RUNTIME.get("default_lakehouse_name", "")
        lh_i = _NB_RUNTIME.get("default_lakehouse_id", "")
        if ws_i or ws_n:
            print(f"   notebook ws    : {ws_n or '(no name)'}  ({ws_i or '?'})")
        if lh_i or lh_n:
            print(f"   default LH     : {lh_n or '(no name)'}  ({lh_i or '?'})")
        else:
            print(f"   default LH     : (none mounted to this notebook)")
    print(f"   scan workspace : {SOURCE_LAKEHOUSE_WORKSPACE_NAME or '(unknown)'}"
          f"  ({SOURCE_LAKEHOUSE_WORKSPACE_ID or '(unknown)'})")
    print(f"   scan lakehouse : {SOURCE_LAKEHOUSE_NAME or '(unknown)'}"
          f"  ({SOURCE_LAKEHOUSE_ID or '(unknown)'})")
    print(f"   path           : {source_uri}")
    print(f"   glob           : {SOURCE_FILE_GLOB}")
elif READ_WORKSPACE_IDS:
    print(f"                    restricted to {len(READ_WORKSPACE_IDS)} workspace(s)")
if MAX_NOTEBOOKS:
    print(f"                    capped at {MAX_NOTEBOOKS} notebooks (test mode)")
print(f" WRITE TO         : {write_kind}")
print(f"   inventory      : {inventory_target}")
print(f"   findings       : {output_target}")
print(f" MIN SEVERITY     : {MIN_SEVERITY}")
print(f" SCAN MD+OUTPUTS  : {SCAN_MARKDOWN_AND_OUTPUTS}")
if SOURCE_MODE == "api":
    print(f" CONCURRENCY      : {EXECUTOR_CONCURRENCY}/executor x "
          f"partition size {TARGET_PARTITION_SIZE}")
else:
    print(f" PARTITIONS       : ~{TARGET_PARTITION_SIZE} files per Spark partition")
print("============================================================")


In [ ]:
# === Imports + token =========================================================
import asyncio
import ast
import base64
import json
import math
import re
import time
from datetime import datetime, timezone
from typing import Iterable, Iterator, List, Tuple

import aiohttp
import requests
from pyspark.sql import Row
from pyspark.sql.types import (
    BooleanType, IntegerType, StringType, StructField, StructType, TimestampType
)

try:
    import notebookutils
    _get_token = lambda aud: notebookutils.credentials.getToken(aud)
except ImportError:
    import mssparkutils
    _get_token = lambda aud: mssparkutils.credentials.getToken(aud)


def get_token() -> str:
    return _get_token(TOKEN_AUDIENCE)


_token = get_token()
print(f"Acquired Fabric token ({len(_token)} chars).")


In [ ]:
# === Scanner engine: patterns, AST, entropy, URLs, cross-workspace ==========
# Driver-side reference. Each executor partition function inlines its own
# copy of these because Spark serialization across workers is most reliable
# when the partition function is self-contained.

SEVERITY_ORDER = ["low", "medium", "high", "critical"]
_MIN_SEV_IDX = SEVERITY_ORDER.index(MIN_SEVERITY.lower())


def _meets_severity(sev):
    try:
        return SEVERITY_ORDER.index(sev) >= _MIN_SEV_IDX
    except ValueError:
        return True


# --- URL schemes (NotebookAudit's wider list) -------------------------------
URL_PROTOCOL_SCHEMES = [
    "http://", "https://", "ftp://", "ftps://", "sftp://",
    "abfs://", "abfss://", "wasb://", "wasbs://", "adl://",
    "s3://", "s3a://", "s3n://", "gs://",
    "hdfs://", "viewfs://", "dbfs://",
    "jdbc:", "odbc:", "mongodb://", "mongodb+srv://",
    "postgresql://", "postgres://", "mysql://", "mssql://",
    "redis://", "rediss://", "amqp://", "amqps://",
    "kafka://", "kinesis://", "eventhubs://",
    "mqtt://", "mqtts://",
    "ssh://", "scp://", "smb://", "nfs://", "tcp://", "udp://",
]

_scheme_prefixes = []
for _s in URL_PROTOCOL_SCHEMES:
    _p = _s.rstrip(":/")
    if _p and _p not in _scheme_prefixes:
        _scheme_prefixes.append(re.escape(_p))
_schemes_alt = "|".join(sorted(_scheme_prefixes, key=len, reverse=True))

URL_RE = re.compile(
    r'(?P<url>(?:' + _schemes_alt + r')s?://[^\s"\'`<>)\]},;]+'
    r'|(?:' + _schemes_alt + r')s?:[^\s"\'`<>)\]},;]+'
    r'|\bwww\.[^\s"\'`<>)\]},;]+\.[^\s"\'`<>)\]},;]+)',
    re.IGNORECASE,
)
TRAILING_CHARS = '.,;:!?)]}>"\'`'

BENIGN_URL_PATTERNS = [
    re.compile(r'https?://(localhost|127\.0\.0\.1|0\.0\.0\.0)(:|/|$)', re.I),
    re.compile(r'https?://json\.schemastore\.org/', re.I),
    re.compile(r'https?://schemas?\.(microsoft|xmlsoap|w3)\.', re.I),
    re.compile(r'https?://(pypi|pip|conda|anaconda)\.(org|io)/', re.I),
    re.compile(r'https?://(docs|learn|aka)\.(microsoft|python|spark)\.', re.I),
    re.compile(r'https?://spark\.apache\.org/', re.I),
    re.compile(r'^file://', re.I),
]

READ_CONTEXT_RE = re.compile(
    r'(?:\brequests?\.(?:get|head|options)\s*\('
    r'|\bspark\.read\b'
    r'|\.read_\w+\s*\('
    r'|\bdownload\w*\s*\('
    r'|!\s*wget\b)',
    re.I,
)
WRITE_CONTEXT_RE = re.compile(
    r'(?:\brequests?\.(?:post|put|patch|delete)\s*\('
    r'|\.write\.\w+'
    r'|\.save\s*\('
    r'|\.to_\w+\s*\('
    r'|\bupload\w*\s*\('
    r'|\.write_\w+\s*\()',
    re.I,
)


# --- Suspicious imports (NotebookAudit) -------------------------------------
SUSPICIOUS_IMPORTS = {
    "requests", "httpx", "urllib", "urllib3", "http.client", "aiohttp", "pycurl",
    "boto3", "botocore", "azure.storage.blob", "azure.storage.filedatalake",
    "google.cloud.storage", "gcsfs", "s3fs", "adlfs",
    "paramiko", "ftplib", "pysftp", "smtplib",
    "socket", "websocket", "websockets",
    "grpc", "grpcio",
    "pymongo", "kafka", "confluent_kafka", "pika",
    "azure.eventhub", "azure.servicebus",
    "telegram", "slack_sdk", "discord", "twilio", "sendgrid",
    "msgraph", "O365", "office365",
    "ctypes", "cffi",
    "dropbox", "boxsdk", "pydrive", "pydrive2", "googleapiclient",
    "mega", "firebase_admin", "pyrebase", "supabase", "cloudinary", "b2sdk",
    "pyperclip", "clipboard",
}

IMPORT_CATEGORIES = {
    "requests": "http_exfiltration", "httpx": "http_exfiltration",
    "urllib": "http_exfiltration", "aiohttp": "http_exfiltration",
    "boto3": "cloud_storage_write", "botocore": "cloud_storage_write",
    "azure.storage.blob": "cloud_storage_write",
    "google.cloud.storage": "cloud_storage_write",
    "paramiko": "filesystem_exfiltration", "ftplib": "filesystem_exfiltration",
    "smtplib": "email_exfiltration",
    "socket": "raw_socket_network", "websocket": "raw_socket_network",
    "pymongo": "external_database_write", "kafka": "external_database_write",
    "telegram": "webhook_exfiltration", "slack_sdk": "webhook_exfiltration",
    "discord": "webhook_exfiltration",
    "msgraph": "api_exfiltration", "O365": "api_exfiltration",
    "dropbox": "cloud_storage_write", "boxsdk": "cloud_storage_write",
    "pydrive2": "cloud_storage_write", "googleapiclient": "cloud_storage_write",
    "mega": "cloud_storage_write", "firebase_admin": "cloud_storage_write",
    "supabase": "cloud_storage_write", "cloudinary": "cloud_storage_write",
    "ctypes": "native_code_execution", "cffi": "native_code_execution",
}


# --- 100+ regex patterns: union of NotebookAudit + v1's secret extensions ---
# Tuple format: (compiled_regex, category, severity, message)
PATTERNS = [
    # --- HTTP exfiltration ---
    (re.compile(r'(requests|httpx|urllib)\s*\.\s*(post|put|patch)\s*\(', re.I),
     "http_exfiltration", "high", "HTTP POST/PUT/PATCH request detected"),
    (re.compile(r'https?://[^\s\'"]+\.(ngrok|burpcollaborator|requestbin|pipedream|webhook\.site|hookbin|canarytokens|interactsh)', re.I),
     "http_exfiltration", "critical", "Known exfiltration service URL"),
    (re.compile(r'https?://\d{1,3}\.\d{1,3}\.\d{1,3}\.\d{1,3}[:/]', re.I),
     "http_exfiltration", "high", "HTTP request to raw IP address"),
    (re.compile(r'urllib\.request\.urlopen\s*\([^,)]+,\s*(data|b["\'])', re.I),
     "http_exfiltration", "high", "urllib.request.urlopen() with data — POST equivalent"),
    (re.compile(r'urllib\.request\.urlretrieve\s*\(|urlretrieve\s*\(', re.I),
     "http_exfiltration", "medium", "urlretrieve() downloads file to local disk"),

    # --- Cloud storage / file sharing services ---
    (re.compile(r'(api\.dropboxapi\.com|content\.dropboxapi\.com|dropbox\.com/oauth2)', re.I),
     "cloud_storage_write", "high", "Dropbox API endpoint"),
    (re.compile(r'\b(dropbox\.Dropbox|DropboxOAuth2FlowNoRedirect)\s*\(', re.I),
     "cloud_storage_write", "high", "Dropbox SDK client initialization"),
    (re.compile(r'(api\.box\.com|upload\.box\.com|box\.com/api)', re.I),
     "cloud_storage_write", "high", "Box.com API endpoint"),
    (re.compile(r'(drive\.google\.com|www\.googleapis\.com/upload/drive|www\.googleapis\.com/drive)', re.I),
     "cloud_storage_write", "high", "Google Drive API endpoint"),
    (re.compile(r'(mega\.nz|mega\.co\.nz|g\.api\.mega\.co\.nz)', re.I),
     "cloud_storage_write", "high", "MEGA cloud storage endpoint"),
    (re.compile(r'(wetransfer\.com/api|we\.tl)', re.I),
     "cloud_storage_write", "high", "WeTransfer API/URL"),
    (re.compile(r'(api\.pcloud\.com|pcloud\.com/oauth2)', re.I),
     "cloud_storage_write", "high", "pCloud API endpoint"),
    (re.compile(r'(firebaseio\.com|firebasestorage\.googleapis\.com)', re.I),
     "cloud_storage_write", "high", "Firebase endpoint"),
    (re.compile(r'\b\w+\.supabase\.co\b', re.I),
     "cloud_storage_write", "high", "Supabase endpoint"),
    (re.compile(r'(api\.cloudinary\.com|cloudinary\.uploader)', re.I),
     "cloud_storage_write", "high", "Cloudinary upload endpoint"),
    (re.compile(r'(backblazeb2\.com|b2_authorize_account|b2_upload_file)', re.I),
     "cloud_storage_write", "high", "Backblaze B2 storage endpoint"),
    (re.compile(r'(icloud\.com/api|setup\.icloud\.com)', re.I),
     "cloud_storage_write", "high", "iCloud API endpoint"),
    (re.compile(r'\b(upload_blob|upload_file|upload_fileobj|upload_from_string|upload_data|put_object|put_block_blob|put_item|put_record)\s*\(', re.I),
     "cloud_storage_write", "high", "Cloud storage upload SDK call"),

    # --- Paste / gist services ---
    (re.compile(r'(pastebin\.com/api|api\.paste\.ee|dpaste\.com/api|ix\.io|sprunge\.us|hastebin\.com)', re.I),
     "api_exfiltration", "high", "Paste service API — data exfiltration risk"),
    (re.compile(r'api\.github\.com/gists', re.I),
     "api_exfiltration", "high", "GitHub Gists API — data exfiltration risk"),
    (re.compile(r'(gitlab\.com/api.*snippets|snippets\.gitlab\.com)', re.I),
     "api_exfiltration", "high", "GitLab Snippets API"),
    (re.compile(r'api\.airtable\.com', re.I),
     "api_exfiltration", "high", "Airtable API endpoint"),
    (re.compile(r'api\.notion\.(com|so)', re.I),
     "api_exfiltration", "high", "Notion API endpoint"),

    # --- Webhooks ---
    (re.compile(r'https://hooks\.slack\.com/services/', re.I),
     "webhook_exfiltration", "critical", "Slack incoming webhook URL"),
    (re.compile(r'https://discord(app)?\.com/(api/)?webhooks/', re.I),
     "webhook_exfiltration", "critical", "Discord webhook URL"),
    (re.compile(r'https://api\.telegram\.org/bot', re.I),
     "webhook_exfiltration", "critical", "Telegram bot API"),
    (re.compile(r'https://[^\s]*\.execute-api\.[^\s]*\.amazonaws\.com', re.I),
     "webhook_exfiltration", "high", "AWS API Gateway endpoint"),
    (re.compile(r'https://[^\s]+(\.workers\.dev|\.vercel\.app|\.netlify\.app|\.pages\.dev|\.azurewebsites\.net)', re.I),
     "webhook_exfiltration", "high", "Serverless platform endpoint"),

    # --- Microsoft API abuse ---
    (re.compile(r'https://graph\.microsoft\.com/v\d+\.\d+/(me|users|groups|drives|sites)', re.I),
     "api_exfiltration", "high", "Microsoft Graph API call"),
    (re.compile(r'https://management\.azure\.com/subscriptions', re.I),
     "api_exfiltration", "high", "Azure Management API call"),
    (re.compile(r'https://api\.fabric\.microsoft\.com/v1/', re.I),
     "cross_workspace_operation", "high", "Fabric REST API call from within notebook"),
    (re.compile(r'https://api\.powerbi\.com/v1\.0/myorg/datasets/[^/]+/rows', re.I),
     "api_exfiltration", "critical", "Power BI streaming/push dataset API"),
    (re.compile(r'(GraphServiceClient|MSGraph|O365\.Account)\s*\(', re.I),
     "api_exfiltration", "critical", "Microsoft Graph SDK client"),

    # --- Credential access (NotebookAudit + v1's union) ---
    (re.compile(r'(mssparkutils|notebookutils)\.credentials\.(getSecret|getToken|getConnectionString|getFullConnectionString)', re.I),
     "credential_access", "high", "Fabric credential access"),
    (re.compile(r'(TokenLibrary|dbutils\.secrets)\.(get|getSecret|getToken)', re.I),
     "credential_access", "high", "Token/secret library access"),
    (re.compile(r'AccountKey\s*=\s*[A-Za-z0-9+/=]{40,}', re.I),
     "credential_access", "critical", "Hardcoded Storage Account Key"),
    (re.compile(r'(\?|&)sig=[A-Za-z0-9%+/=]{20,}', re.I),
     "credential_access", "critical", "SAS token with signature"),
    (re.compile(r'["\']https?://[^"\s]+\.blob\.core\.windows\.net[^"\s]*sig=[^"\s]+["\']', re.I),
     "credential_access", "critical", "SAS-tokened blob URL assignment"),
    (re.compile(r'(Password|pwd)\s*=\s*[^;\s"]{4,}', re.I),
     "credential_access", "high", "Hardcoded SQL password"),
    (re.compile(r'(client[_-]?secret|app[_-]?secret)\s*[=:]\s*["\']?[A-Za-z0-9~._-]{16,}', re.I),
     "credential_access", "critical", "Hardcoded client/app secret"),
    (re.compile(r'["\']Bearer\s+eyJ[A-Za-z0-9_-]{20,}', re.I),
     "credential_access", "critical", "Hardcoded Bearer JWT token"),
    (re.compile(r'SharedAccessKey\s*=\s*[A-Za-z0-9+/=]{30,}', re.I),
     "credential_access", "critical", "Hardcoded Event Hub / Service Bus key"),
    (re.compile(r'(openai[_-]?api[_-]?key|OPENAI_API_KEY)\s*[=:]\s*["\']?sk-[A-Za-z0-9]{20,}', re.I),
     "credential_access", "critical", "Hardcoded OpenAI API key"),
    (re.compile(r'(azure[_-]?openai[_-]?key|AZURE_OPENAI_KEY)\s*[=:]\s*["\']?[A-Fa-f0-9]{32}', re.I),
     "credential_access", "critical", "Hardcoded Azure OpenAI key"),
    (re.compile(r'(AccountKey|cosmosdb[_-]?key)\s*[=:]\s*["\']?[A-Za-z0-9+/=]{40,}', re.I),
     "credential_access", "critical", "Hardcoded Cosmos DB / Storage key"),
    (re.compile(r'(api[_-]?key|apikey|subscription[_-]?key|Ocp-Apim-Subscription-Key)\s*[=:]\s*["\']?[A-Za-z0-9]{16,}', re.I),
     "credential_access", "high", "Hardcoded API key"),
    (re.compile(r'(connection[_-]?string|conn[_-]?str)\s*[=:]\s*["\'][^"]{20,}["\']', re.I),
     "credential_access", "high", "Hardcoded connection string"),
    (re.compile(r'(secret|password|passwd|token|credential)\s*[=:]\s*["\'][^"\s]{8,}["\']', re.I),
     "credential_access", "medium", "Generic secret assignment"),
    (re.compile(r'spark\.conf\.set\s*\([^)]*account\.key[^)]*\)', re.I),
     "credential_access", "high", "Spark config with storage account key"),
    (re.compile(r'spark\.conf\.set\s*\(\s*["\'].*(account\.key|access\.key|secret|password|token)', re.I),
     "credential_access", "high", "Spark config with external credentials"),
    (re.compile(r'jdbc:[a-z]+://[^\s]*password=[^;\s"]{4,}', re.I),
     "credential_access", "critical", "JDBC URL with embedded password"),

    # --- Shell / package install ---
    (re.compile(r'^!pip\s+install\b|^%pip\s+install\b|^%conda\s+install\b', re.I | re.MULTILINE),
     "in_notebook_package_install", "high", "In-notebook package installation"),
    (re.compile(r'pip\s+install\s+.*--(index-url|extra-index-url)\s+http', re.I),
     "in_notebook_package_install", "critical", "pip install from non-PyPI index"),
    (re.compile(r'^!(curl|wget|nc|ncat|netcat|scp|rsync|ssh|sftp|ftp)\b', re.I | re.MULTILINE),
     "shell_execution", "critical", "Shell command for network file transfer"),
    (re.compile(r'(os\.system|os\.popen|subprocess\.(run|Popen|call|check_output))\s*\(', re.I),
     "shell_execution", "high", "Shell command execution"),

    # --- Spark external writes ---
    (re.compile(r'\.format\s*\(\s*["\'](?:com\.mongodb|org\.apache\.spark\.sql\.cassandra|kafka|es\.spark)', re.I),
     "spark_external_write", "high", "Spark write with external connector format"),
    (re.compile(r'writeStream.*\.format\s*\(\s*["\'](?:kafka|eventhubs|socket|mqtt|kinesis)["\']\s*\)', re.I),
     "spark_streaming_exfiltration", "critical", "Spark streaming to external broker"),
    (re.compile(r'writeStream.*foreachBatch', re.I),
     "spark_streaming_exfiltration", "high", "Spark streaming foreachBatch — custom sink"),
    (re.compile(r'(dbutils\.fs\.cp|mssparkutils\.fs\.cp|shutil\.copy).*(?:s3|wasb|gs|abfss|adl|ftp)', re.I),
     "filesystem_exfiltration", "critical", "File copy to external cloud storage"),
    (re.compile(r'(dbutils\.fs\.mount|mssparkutils\.fs\.mount)', re.I),
     "spark_external_write", "high", "Mounting external storage"),
    (re.compile(r'(mssparkutils|notebookutils)\.fs\.(put|append|mv)\s*\(', re.I),
     "filesystem_exfiltration", "high", "mssparkutils/notebookutils file write/move"),

    # --- JDBC / database ---
    (re.compile(r'jdbc:(mysql|postgresql|sqlserver|oracle|db2|redshift|snowflake|bigquery)://[^\s\'"]+', re.I),
     "external_database_write", "high", "JDBC connection string to external database"),
    (re.compile(r'Server\s*=\s*[^;]+\.database\.windows\.net', re.I),
     "external_database_write", "high", "Azure SQL connection string"),

    # --- Email ---
    (re.compile(r'(smtplib\.SMTP|MIMEMultipart|MIMEText|send_message|sendmail)\s*[(\.]', re.I),
     "email_exfiltration", "high", "Email sending capability"),
    (re.compile(r'(api\.sendgrid\.com|api\.mailgun\.net|mandrillapp\.com/api|api\.sparkpost\.com)', re.I),
     "email_exfiltration", "high", "Transactional email API endpoint"),

    # --- DNS exfiltration ---
    (re.compile(r'(dns\.resolver|dnspython|socket\.getaddrinfo)', re.I),
     "dns_exfiltration", "high", "DNS resolution — potential DNS exfiltration"),

    # --- Encoding / obfuscation ---
    (re.compile(r'(exec|eval)\s*\(\s*(base64|codecs|bytes\.fromhex|bytearray)', re.I),
     "dynamic_code_execution", "critical", "Execution of decoded/deobfuscated code"),
    (re.compile(r'getattr\s*\(\s*__builtins__', re.I),
     "dynamic_code_execution", "critical", "Dynamic access to builtins — obfuscation"),

    # --- Polars / Pandas writes ---
    (re.compile(r'\.write_(?:delta|parquet|csv|json|ndjson|ipc|arrow|avro|excel|database)\s*\(', re.I),
     "filesystem_exfiltration", "medium", "Polars DataFrame write to file/database"),
    (re.compile(r'\.to_(?:csv|json|parquet|sql|excel|html|hdf|pickle|orc|feather|stata|gbq|xml)\s*\(', re.I),
     "filesystem_exfiltration", "medium", "Pandas DataFrame export to file/database"),

    # --- SQL DML ---
    (re.compile(r'\.execute\s*\(\s*[f"\']*(INSERT\s+INTO|UPDATE\s+|DELETE\s+FROM|MERGE\s+INTO)', re.I),
     "external_database_write", "medium", "SQL DML via .execute()"),
    (re.compile(r'spark\.sql\s*\(\s*f?["\']*(INSERT\s+INTO|UPDATE\s+|DELETE\s+FROM|MERGE\s+INTO)', re.I),
     "spark_external_write", "medium", "spark.sql() with data-modifying statement"),

    # --- Fabric APIs that need migration / update review ---
    # fastcp is Fabric's high-throughput bulk file copy utility. Any notebook
    # using it deserves a migration / review pass: the source/destination URIs
    # are usually OneLake or external storage, and the API surface has shifted
    # from mssparkutils.fs.fastcp -> notebookutils.fs.fastcp in Fabric.
    (re.compile(r'\b(?:mssparkutils|notebookutils)\.fs\.fastcp\s*\(', re.I),
     "needs_update", "medium",
     "fastcp() bulk copy detected - flag for migration / review"),
    (re.compile(r'(?<!\.)\bfastcp\s*\(', re.I),
     "needs_update", "medium",
     "fastcp() bulk copy detected (bare call) - flag for migration / review"),
    (re.compile(r'from\s+(?:mssparkutils|notebookutils)\.fs\s+import\s+[^\n#]*\bfastcp\b', re.I),
     "needs_update", "medium",
     "fastcp imported - flag for migration / review"),
]


# --- Potential-write (v1): destination-agnostic write API detection. -------
# Catches notebooks that build the destination at runtime (the URL never
# appears as a literal). Surfaced as severity=medium category=potential_write.
WRITE_CALL_RE = re.compile(
    r"\.write_(?:delta|parquet|csv|json|ndjson|ipc|arrow|avro|excel|database)\s*\("
    r"|\.to_(?:csv|json|parquet|sql|excel|html|hdf|pickle|orc|feather|stata|gbq|xml)\s*\("
    r"|\.write\.(?:format|mode|option|partitionBy|saveAsTable|save|insertInto|jdbc)\s*\("
    r"|\.writeStream\.(?:format|trigger|option|outputMode|start|toTable|foreachBatch)\s*\("
    r"|\brequests?\.(?:post|put|patch|delete)\s*\("
    r"|\bhttpx?\.(?:post|put|patch|delete)\s*\("
    r"|\b(?:upload_blob|upload_file|upload_fileobj|upload_from_string|"
    r"upload_data|put_object|put_block_blob|put_item|put_record)\s*\("
    r"|\bmssparkutils\.fs\.(?:put|append|mv|cp|fastcp)\s*\("
    r"|\bnotebookutils\.fs\.(?:put|append|mv|cp|fastcp)\s*\("
    r"|\bdbutils\.fs\.(?:put|cp|mv)\s*\("
    r"|\.execute\s*\(\s*[fbur]*['\"]?(?:INSERT|UPDATE|DELETE|MERGE)\s"
    r"|\bspark\.sql\s*\(\s*f?['\"]?(?:INSERT|UPDATE|DELETE|MERGE)\s"
    r"|\bopen\s*\([^,)]*,\s*['\"][wax]b?",
    re.I,
)


# --- Cross-workspace destination parsing (v1) ------------------------------
WORKSPACE_URL_RE = re.compile(
    r"(?:"
    r"abfss?://([^@/\s]+)@onelake\.dfs\.fabric\.microsoft\.com"
    r"|https?://[^/\s]*\.fabric\.microsoft\.com/(?:[^?\s]*?/)?(?:v1/)?workspaces/([0-9a-f-]{8,})"
    r"|https?://(?:app|api)\.powerbi\.com/(?:[^/\s]*/)*?groups/([0-9a-f-]{8,})"
    r")",
    re.IGNORECASE,
)
GUID_RE = re.compile(
    r"^[0-9a-f]{8}-?[0-9a-f]{4}-?[0-9a-f]{4}-?[0-9a-f]{4}-?[0-9a-f]{12}$",
    re.IGNORECASE,
)


def parse_dest_workspace(url):
    if not url:
        return None
    m = WORKSPACE_URL_RE.search(url)
    if not m:
        return None
    return m.group(1) or m.group(2) or m.group(3)


def resolve_dest_workspace(url, src_ws_id, src_ws_name,
                           ws_name_by_id=None, ws_id_by_name=None):
    raw = parse_dest_workspace(url)
    if not raw:
        return None, None, None
    name_map = ws_name_by_id or {}
    id_map = ws_id_by_name or {}
    if GUID_RE.match(raw):
        dest_id = raw.lower()
        dest_name = name_map.get(dest_id, "") or name_map.get(raw, "")
    else:
        dest_name = raw
        dest_id = id_map.get(raw.lower(), "")
    src_id_l = (src_ws_id or "").lower()
    src_name_l = (src_ws_name or "").lower()
    if dest_id and src_id_l and GUID_RE.match(src_id_l or ""):
        cross = (dest_id != src_id_l)
    elif dest_name and src_name_l:
        cross = (dest_name.lower() != src_name_l)
    else:
        cross = None
    return (dest_id or None), (dest_name or None), cross


# --- Entropy analysis (NotebookAudit) --------------------------------------
def _shannon_entropy(s):
    if not s:
        return 0.0
    freq = {}
    for c in s:
        freq[c] = freq.get(c, 0) + 1
    n = len(s)
    return -sum((count / n) * math.log2(count / n) for count in freq.values())


# --- Snippet redaction for credentials (v1) --------------------------------
def _redact(s, keep=4):
    if not s:
        return s
    s = s.strip()
    if len(s) <= keep * 2 + 3:
        return "***"
    return s[:keep] + "*" * (len(s) - keep * 2) + s[-keep:]


def _trim_snippet(text, max_bytes):
    t = (text or "").strip()
    if len(t) <= max_bytes:
        return t
    return t[: max_bytes - 1] + "…"


# --- AST: suspicious imports + eval/exec (NotebookAudit) -------------------
def _ast_scan(code_str):
    findings = []
    try:
        tree = ast.parse(code_str)
    except SyntaxError:
        return findings
    for node in ast.walk(tree):
        if isinstance(node, (ast.Import, ast.ImportFrom)):
            mods = ([a.name for a in node.names] if isinstance(node, ast.Import)
                    else ([node.module] if node.module else []))
            for mod in mods:
                if not mod:
                    continue
                top = mod.split(".")[0]
                matched = mod if mod in SUSPICIOUS_IMPORTS else (
                    top if top in SUSPICIOUS_IMPORTS else None)
                if matched:
                    cat = IMPORT_CATEGORIES.get(matched, "suspicious_import")
                    findings.append({
                        "finding_type": "import",
                        "category": cat, "severity": "high",
                        "message": f"Import of suspicious module: {mod}",
                        "line": getattr(node, "lineno", 0),
                    })
        if isinstance(node, ast.Call):
            if isinstance(node.func, ast.Name) and node.func.id in ("eval", "exec", "compile"):
                findings.append({
                    "finding_type": "dynamic_exec",
                    "category": "dynamic_code_execution", "severity": "critical",
                    "message": f"{node.func.id}() call — dynamic code execution",
                    "line": node.lineno,
                })
    return findings


# --- Multi-format code extractor (NotebookAudit, extended for markdown) ---
def _extract_blocks(content_bytes, file_path, include_md_and_outputs=True):
    """Yields list of (text, cell_index, source_kind, part_path).

    source_kind ∈ {"code", "markdown", "output", "raw"}.
    part_path is "" for plain .ipynb / .py inputs; for Fabric Item JSON it is
    the path of the part inside `definition.parts` (e.g. "notebook-content.py").
    """
    try:
        text = content_bytes.decode("utf-8", errors="ignore") if isinstance(
            content_bytes, (bytes, bytearray)) else str(content_bytes)
    except Exception:
        return []

    fp = (file_path or "").lower()
    if fp.endswith(".py"):
        return [(text, 0, "code", "")]
    if fp.endswith(".md"):
        return [(text, 0, "markdown", "")]

    if fp.endswith(".ipynb") or fp.endswith(".json") or text.lstrip().startswith("{"):
        try:
            data = json.loads(text)
        except (json.JSONDecodeError, ValueError):
            return [(text, 0, "raw", "")]
        if not isinstance(data, dict):
            return [(text, 0, "raw", "")]

        blocks = []

        # Standard .ipynb
        if "cells" in data and isinstance(data["cells"], list):
            for i, cell in enumerate(data["cells"]):
                ct = cell.get("cell_type")
                src = cell.get("source", [])
                txt = "".join(src) if isinstance(src, list) else str(src)
                if ct == "code":
                    blocks.append((txt, i, "code", ""))
                elif ct == "markdown" and include_md_and_outputs:
                    blocks.append((txt, i, "markdown", ""))
                if include_md_and_outputs:
                    for out in cell.get("outputs", []) or []:
                        if not isinstance(out, dict):
                            continue
                        t = out.get("text", "")
                        if t:
                            ot = "".join(t) if isinstance(t, list) else str(t)
                            blocks.append((ot, i, "output", ""))
                        # Stream / display_data text/plain
                        data_field = out.get("data") or {}
                        for key, val in data_field.items():
                            if "text" in key.lower():
                                vt = "".join(val) if isinstance(val, list) else str(val)
                                blocks.append((vt, i, "output", ""))
            return blocks

        # Fabric Item JSON: definition.parts (base64-encoded payloads)
        parts = (data.get("definition") or {}).get("parts") or []
        if parts:
            for i, part in enumerate(parts):
                path = part.get("path", "") or ""
                payload = part.get("payload", "") or ""
                if not payload:
                    continue
                try:
                    decoded = base64.b64decode(payload).decode("utf-8", errors="ignore")
                except Exception:
                    continue
                if path.endswith(".ipynb"):
                    try:
                        sub = json.loads(decoded)
                    except (json.JSONDecodeError, ValueError):
                        blocks.append((decoded, i, "raw", path))
                        continue
                    for j, cell in enumerate(sub.get("cells", []) or []):
                        ct = cell.get("cell_type")
                        src = cell.get("source", [])
                        txt = "".join(src) if isinstance(src, list) else str(src)
                        if ct == "code":
                            blocks.append((txt, j, "code", path))
                        elif ct == "markdown" and include_md_and_outputs:
                            blocks.append((txt, j, "markdown", path))
                        if include_md_and_outputs:
                            for out in cell.get("outputs", []) or []:
                                if not isinstance(out, dict):
                                    continue
                                t = out.get("text", "")
                                if t:
                                    ot = "".join(t) if isinstance(t, list) else str(t)
                                    blocks.append((ot, j, "output", path))
                elif path.endswith(".py"):
                    blocks.append((decoded, i, "code", path))
                elif path.endswith(".md"):
                    if include_md_and_outputs:
                        blocks.append((decoded, i, "markdown", path))
                else:
                    if include_md_and_outputs:
                        blocks.append((decoded, i, "raw", path))
            if blocks:
                return blocks

        # Fabric Item JSON: properties.cells
        cells = (data.get("properties") or {}).get("cells") or []
        if cells:
            for i, cell in enumerate(cells):
                ct = cell.get("cell_type")
                src = cell.get("source", [])
                txt = "".join(src) if isinstance(src, list) else str(src)
                if ct == "code":
                    blocks.append((txt, i, "code", ""))
                elif ct == "markdown" and include_md_and_outputs:
                    blocks.append((txt, i, "markdown", ""))
            return blocks

    return [(text, 0, "raw", "")]


def _scan_one_block(text, cell_idx, source_kind, code_cells_only_for_pattern=True):
    """Run all detectors on one block. Returns a list of finding dicts."""
    findings = []
    if not text or not text.strip():
        return findings
    lines = text.splitlines()

    is_code = (source_kind == "code")

    # 1. Regex PATTERNS (only on code cells)
    if is_code:
        for pattern, category, severity, message in PATTERNS:
            for m in pattern.finditer(text):
                if not _meets_severity(severity):
                    continue
                line_no = text[: m.start()].count("\n") + 1
                snippet_line = (lines[line_no - 1] if line_no <= len(lines) else "").strip()
                snippet = _trim_snippet(snippet_line, MAX_SNIPPET_BYTES)
                if category == "credential_access":
                    snippet = _redact(snippet, keep=4)
                findings.append({
                    "finding_type": "pattern",
                    "category": category, "severity": severity,
                    "message": message, "line": line_no,
                    "code_snippet": snippet,
                    "url": None, "direction": None,
                })

    # 2. AST (only on code cells)
    if is_code:
        for f in _ast_scan(text):
            if _meets_severity(f["severity"]):
                line = f.get("line", 0)
                snippet_line = (lines[line - 1] if 0 < line <= len(lines) else "").strip()
                f["code_snippet"] = _trim_snippet(snippet_line, MAX_SNIPPET_BYTES)
                f["url"] = None
                f["direction"] = None
                findings.append(f)

    # 3. URL detection (code, markdown, outputs)
    seen_urls = set()
    url_severity = "medium" if is_code else "low"
    if _meets_severity(url_severity):
        for m in URL_RE.finditer(text):
            url = m.group("url")
            while url and url[-1] in TRAILING_CHARS:
                url = url[:-1]
            if not url:
                continue
            ul = url.lower().rstrip("/")
            if ul in seen_urls:
                continue
            if any(bp.search(url) for bp in BENIGN_URL_PATTERNS):
                continue
            seen_urls.add(ul)
            if is_code:
                ctx_start = max(0, m.start() - 300)
                ctx = text[ctx_start: m.start()]
                has_read = READ_CONTEXT_RE.search(ctx)
                has_write = WRITE_CONTEXT_RE.search(ctx)
                if has_read and has_write:
                    direction = "read_write"
                elif has_write:
                    direction = "write"
                elif has_read:
                    direction = "read"
                else:
                    direction = "unknown"
            else:
                direction = "reference"
            line_no = text[: m.start()].count("\n") + 1
            snippet_line = (lines[line_no - 1] if line_no <= len(lines) else "").strip()
            findings.append({
                "finding_type": "url",
                "category": "external_url_reference",
                "severity": url_severity,
                "message": f"External URL/URI [direction: {direction}]: {url[:100]}",
                "line": line_no,
                "code_snippet": _trim_snippet(snippet_line, MAX_SNIPPET_BYTES),
                "url": url, "direction": direction,
            })

    # 4. Entropy (only on code cells)
    if is_code and _meets_severity("high"):
        try:
            tree = ast.parse(text)
            for node in ast.walk(tree):
                if isinstance(node, ast.Constant) and isinstance(node.value, str):
                    s = node.value
                    if len(s) > 40:
                        ent = _shannon_entropy(s)
                        if ent > 4.5 and not s.lstrip().upper().startswith(
                                ("SELECT", "INSERT", "CREATE", "ALTER", "UPDATE", "DELETE", "MERGE", "WITH")):
                            line = getattr(node, "lineno", 0)
                            findings.append({
                                "finding_type": "entropy",
                                "category": "encoding_obfuscation",
                                "severity": "high",
                                "message": f"High-entropy string literal (entropy={ent:.2f}, len={len(s)})",
                                "line": line,
                                "code_snippet": _trim_snippet(s, MAX_SNIPPET_BYTES),
                                "url": None, "direction": None,
                            })
        except SyntaxError:
            pass

    # 5. Potential-write (v1): destination-agnostic write call detection.
    if is_code and _meets_severity("medium"):
        seen_pw = set()
        for m in WRITE_CALL_RE.finditer(text):
            sig = m.group(0)
            line_no = text[: m.start()].count("\n") + 1
            snippet_line = (lines[line_no - 1] if line_no <= len(lines) else "").strip()
            key = (line_no, sig.strip())
            if key in seen_pw:
                continue
            seen_pw.add(key)
            findings.append({
                "finding_type": "potential_write",
                "category": "potential_write",
                "severity": "medium",
                "message": f"Write-API call detected: {sig.strip()[:60]}",
                "line": line_no,
                "code_snippet": _trim_snippet(snippet_line, MAX_SNIPPET_BYTES),
                "url": None, "direction": "potential_write",
            })

    return findings


def _extract_attached_lakehouse(content_bytes, file_label):
    """Inspects the notebook's metadata to find its default ("attached")
    Lakehouse. Looks at metadata.dependencies.lakehouse — the standard
    Fabric/Synapse notebook attachment block.

    Returns a dict with keys:
        attached_lakehouse_id
        attached_lakehouse_name
        attached_lakehouse_workspace_id
        attached_lakehouse_workspace_name  (always None here; resolved by the
                                            executor via the broadcast ws map)
    Any field that cannot be determined is None.
    Handles plain .ipynb, Fabric Item JSON with definition.parts (base64
    .ipynb payloads), and Fabric Item JSON with properties.metadata.
    """
    empty = {
        "attached_lakehouse_id": None,
        "attached_lakehouse_name": None,
        "attached_lakehouse_workspace_id": None,
        "attached_lakehouse_workspace_name": None,
    }
    try:
        text = content_bytes.decode("utf-8", errors="ignore") if isinstance(
            content_bytes, (bytes, bytearray)) else str(content_bytes)
    except Exception:
        return dict(empty)

    fp = (file_label or "").lower()
    if fp.endswith(".py") or fp.endswith(".md"):
        return dict(empty)

    def _from_meta(meta):
        if not isinstance(meta, dict):
            return None
        deps = meta.get("dependencies")
        if not isinstance(deps, dict):
            return None
        lh = deps.get("lakehouse")
        if not isinstance(lh, dict):
            return None
        lh_id = lh.get("default_lakehouse") or lh.get("default_lakehouse_id")
        lh_name = lh.get("default_lakehouse_name")
        lh_wsid = lh.get("default_lakehouse_workspace_id")
        # Some exports nest known_lakehouses[0] when default_* is missing.
        if not lh_id:
            known = lh.get("known_lakehouses") or []
            if isinstance(known, list) and known and isinstance(known[0], dict):
                lh_id = known[0].get("id") or known[0].get("lakehouse_id")
        if not (lh_id or lh_name or lh_wsid):
            return None
        return {
            "attached_lakehouse_id": lh_id or None,
            "attached_lakehouse_name": lh_name or None,
            "attached_lakehouse_workspace_id": lh_wsid or None,
            "attached_lakehouse_workspace_name": None,
        }

    try:
        data = json.loads(text)
    except (json.JSONDecodeError, ValueError):
        return dict(empty)
    if not isinstance(data, dict):
        return dict(empty)

    # Plain .ipynb
    result = _from_meta(data.get("metadata"))
    if result:
        return result

    # Fabric Item JSON: definition.parts (base64-encoded .ipynb payloads)
    parts = (data.get("definition") or {}).get("parts") or []
    for part in parts:
        path = part.get("path", "") or ""
        payload = part.get("payload", "") or ""
        if not payload or not path.endswith(".ipynb"):
            continue
        try:
            decoded = base64.b64decode(payload).decode("utf-8", errors="ignore")
            sub = json.loads(decoded)
        except Exception:
            continue
        result = _from_meta(sub.get("metadata"))
        if result:
            return result

    # Fabric Item JSON: properties.metadata
    props = data.get("properties")
    if isinstance(props, dict):
        result = _from_meta(props.get("metadata"))
        if result:
            return result

    return dict(empty)


def scan_notebook_bytes(content_bytes, file_label,
                        include_md_and_outputs=True):
    """Top-level driver scan. Returns list of finding dicts, each annotated
    with cell_index, source_kind, part_path, and attached_lakehouse_*.
    URL findings carry no dest_workspace yet (the executor adds that). Same
    for attached_lakehouse_workspace_name (executor resolves via broadcast)."""
    out = []
    lh = _extract_attached_lakehouse(content_bytes, file_label)
    for text, cell_idx, source_kind, part_path in _extract_blocks(
            content_bytes, file_label, include_md_and_outputs):
        for f in _scan_one_block(text, cell_idx, source_kind):
            f["cell_index"] = int(cell_idx)
            f["source_kind"] = source_kind
            f["part_path"] = part_path or ""
            f.update(lh)
            out.append(f)
    return out


print(f"Loaded {len(PATTERNS)} regex patterns "
      f"across {len(set(p[1] for p in PATTERNS))} categories.")
print(f"Suspicious-import list: {len(SUSPICIOUS_IMPORTS)} entries.")
print(f"URL schemes detected: {len(URL_PROTOCOL_SCHEMES)}.")


In [ ]:
# === Source diagnostics ======================================================
# In API mode, probe each enumeration endpoint with the current token.
# In Lakehouse mode, probe the source path to confirm it exists and matches.

if SOURCE_MODE == "lakehouse":
    print(f"[Lakehouse mode] Source: {source_uri}  (glob: {SOURCE_FILE_GLOB})")
    if SOURCE_LAKEHOUSE_WORKSPACE_ID or SOURCE_LAKEHOUSE_ID:
        print(f"  Source workspace : {SOURCE_LAKEHOUSE_WORKSPACE_NAME or '(?)'}"
              f"  ({SOURCE_LAKEHOUSE_WORKSPACE_ID or '(?)'})")
        print(f"  Source lakehouse : {SOURCE_LAKEHOUSE_NAME or '(?)'}"
              f"  ({SOURCE_LAKEHOUSE_ID or '(?)'})")
        if _NB_RUNTIME and SOURCE_LAKEHOUSE_ID == _NB_RUNTIME.get(
                "default_lakehouse_id"):
            print("    (auto-detected from notebookutils.runtime.context — "
                  "this is the lakehouse currently mounted to the notebook)")
    else:
        print("  [WARN] No source workspace/lakehouse known. Output rows will "
              "have empty workspace_id / source_lakehouse_id columns. Either "
              "mount a default lakehouse on this notebook or set "
              "SOURCE_LAKEHOUSE_* in cell 1.")
    try:
        try:
            import notebookutils as _nbu
            _ls = _nbu.fs.ls(source_uri)
        except Exception:
            import mssparkutils as _nbu  # noqa: F401
            _ls = _nbu.fs.ls(source_uri)
        files = [e for e in _ls if not getattr(e, "isDir", False)]
        dirs  = [e for e in _ls if getattr(e, "isDir", False)]
        print(f"  Top-level entries: {len(_ls)}  "
              f"({len(files)} files, {len(dirs)} folders)")
        for e in list(_ls)[:5]:
            kind = "DIR " if getattr(e, "isDir", False) else "FILE"
            print(f"    {kind}  {getattr(e, 'name', e)}")
    except Exception as e:
        print(f"  [ERR] Could not list {source_uri}: {type(e).__name__}: {e}")
else:
    import requests as _req
    _diag_headers = {"Authorization": f"Bearer {_token}",
                     "Content-Type": "application/json"}

    def _probe(label, method, url, **kw):
        try:
            r = _req.request(method, url, headers=_diag_headers,
                             timeout=30, **kw)
            sample = ""
            try:
                j = r.json()
                if isinstance(j, dict):
                    if "value" in j and isinstance(j["value"], list):
                        sample = f"value[]: {len(j['value'])} rows"
                    else:
                        sample = ", ".join(list(j.keys())[:6])
                else:
                    sample = str(j)[:120]
            except Exception:
                sample = r.text[:120]
            print(f"  [{r.status_code}] {label:34s}  {sample}")
            return r.status_code
        except Exception as e:
            print(f"  [ERR] {label:34s}  {type(e).__name__}: {e}")
            return None

    print("Probing endpoints with current identity ...")
    _probe("PBI admin /myorg/admin/groups",
           "GET", f"{PBI_BASE}/v1.0/myorg/admin/groups",
           params={"$top": 1})
    _probe("Fabric admin /v1/admin/workspaces",
           "GET", f"{FABRIC_BASE}/v1/admin/workspaces")
    _probe("Fabric admin /v1/admin/items",
           "GET", f"{FABRIC_BASE}/v1/admin/items")
    _probe("Fabric user /v1/workspaces",
           "GET", f"{FABRIC_BASE}/v1/workspaces")
    print("Done. 200 = accessible; 401/403 = role mismatch; 404 = wrong path.")


In [ ]:
# === Enumerate workspaces + notebooks ========================================
# Strategy:
#   1. List workspaces.  Try in order until one returns >0 rows:
#         a) PBI admin /v1.0/myorg/admin/groups      (legacy, stable)
#         b) Fabric admin /v1/admin/workspaces       (no type filter)
#         c) Fabric /v1/workspaces                   (user / SP membership)
#   2. List notebooks per workspace, in parallel, using whichever path matches
#      the caller's role (admin items endpoint vs. user items endpoint).
# Every step prints HTTP status + counts so you can see exactly where it stops.


async def _http_json(session, method, url, **kw):
    """One request. Returns (status, body_json_or_text, headers). Never raises
    for HTTP-level errors so the caller can decide whether to fall back."""
    async with session.request(method, url, **kw) as r:
        try:
            body = await r.json(content_type=None)
        except Exception:
            body = await r.text()
        return r.status, body, dict(r.headers)


async def _list_pbi_admin_workspaces(session):
    """GET https://api.powerbi.com/v1.0/myorg/admin/groups?$top=5000&$skip=N"""
    out, skip, page_size = [], 0, 5000
    while True:
        params = {"$top": page_size, "$skip": skip}
        status, body, _ = await _http_json(
            session, "GET",
            f"{PBI_BASE}/v1.0/myorg/admin/groups",
            params=params,
        )
        if status != 200:
            raise RuntimeError(f"PBI admin/groups HTTP {status}: {str(body)[:300]}")
        page = body.get("value", []) if isinstance(body, dict) else []
        out.extend(page)
        if len(page) < page_size:
            break
        skip += page_size
    return out


async def _list_fabric_admin_workspaces(session):
    """GET https://api.fabric.microsoft.com/v1/admin/workspaces  (no filters)"""
    out, url = [], f"{FABRIC_BASE}/v1/admin/workspaces"
    while url:
        status, body, _ = await _http_json(session, "GET", url)
        if status != 200:
            raise RuntimeError(f"Fabric admin/workspaces HTTP {status}: {str(body)[:300]}")
        out.extend((body or {}).get("value", []))
        url = (body or {}).get("continuationUri")
    return out


async def _list_user_workspaces(session):
    """GET https://api.fabric.microsoft.com/v1/workspaces"""
    out, url = [], f"{FABRIC_BASE}/v1/workspaces"
    while url:
        status, body, _ = await _http_json(session, "GET", url)
        if status != 200:
            raise RuntimeError(f"User /workspaces HTTP {status}: {str(body)[:300]}")
        out.extend((body or {}).get("value", []))
        url = (body or {}).get("continuationUri")
    return out


async def _list_admin_items_tenant(session):
    """GET /v1/admin/items — tenant-wide items, paged. Filter notebook client-side."""
    out, url = [], f"{FABRIC_BASE}/v1/admin/items"
    first_sample = None
    while url:
        status, body, _ = await _http_json(session, "GET", url)
        if status != 200:
            raise RuntimeError(f"Fabric admin/items HTTP {status}: {str(body)[:300]}")
        items = (body or {}).get("itemEntities") or (body or {}).get("value") or []
        if first_sample is None and items:
            first_sample = items[:3]
        for it in items:
            it_type = (it.get("type") or it.get("itemType") or "").lower()
            if it_type == "notebook":
                ws_id = (it.get("workspaceId")
                         or (it.get("workspace") or {}).get("id"))
                if ws_id:
                    out.append({"workspaceId": ws_id, "id": it["id"],
                                "displayName": it.get("displayName") or it.get("name")})
        url = (body or {}).get("continuationUri")
    return out, first_sample


async def _list_admin_items_workspace(session, wid):
    """GET /v1/admin/items?workspaceId={wid} — per-workspace, paged."""
    out, url = [], f"{FABRIC_BASE}/v1/admin/items?workspaceId={wid}"
    first_sample = None
    while url:
        status, body, _ = await _http_json(session, "GET", url)
        if status != 200:
            return None, status, None
        items = (body or {}).get("itemEntities") or (body or {}).get("value") or []
        if first_sample is None and items:
            first_sample = items[:3]
        for it in items:
            it_type = (it.get("type") or it.get("itemType") or "").lower()
            if it_type == "notebook":
                out.append({"workspaceId": wid, "id": it["id"],
                            "displayName": it.get("displayName") or it.get("name")})
        url = (body or {}).get("continuationUri")
    return out, 200, first_sample


async def _list_user_items_workspace(session, wid):
    """GET /v1/workspaces/{wid}/items — user items, paged, filter client-side."""
    out, url = [], f"{FABRIC_BASE}/v1/workspaces/{wid}/items"
    first_sample = None
    while url:
        status, body, _ = await _http_json(session, "GET", url)
        if status != 200:
            return None, status, None
        items = (body or {}).get("value", [])
        if first_sample is None and items:
            first_sample = items[:3]
        for it in items:
            it_type = (it.get("type") or "").lower()
            if it_type == "notebook":
                out.append({"workspaceId": wid, "id": it["id"],
                            "displayName": it.get("displayName")})
        url = (body or {}).get("continuationUri")
    return out, 200, first_sample


async def run_enumeration(token):
    headers = {"Authorization": f"Bearer {token}",
               "Content-Type": "application/json"}
    timeout = aiohttp.ClientTimeout(total=900, connect=30)
    async with aiohttp.ClientSession(headers=headers, timeout=timeout) as session:

        # ---- Step 1: list workspaces ----------------------------------------
        used_admin = False
        workspaces = []
        if ADMIN_MODE:
            chain = [
                ("PBI admin groups",       _list_pbi_admin_workspaces, True),
                ("Fabric admin workspaces", _list_fabric_admin_workspaces, True),
                ("User /v1/workspaces",    _list_user_workspaces, False),
            ]
        else:
            chain = [("User /v1/workspaces", _list_user_workspaces, False)]

        for label, fn, is_admin in chain:
            try:
                ws = await fn(session)
                print(f"  [{label}] returned {len(ws)} workspaces.")
                if ws:
                    workspaces = ws
                    used_admin = is_admin
                    break
            except Exception as e:
                print(f"  [{label}] FAILED: {e}")

        ws_ids = [w["id"] for w in workspaces]
        # Build a stable id -> name map so each notebook can carry its
        # workspace's human-readable name through to the output table.
        ws_name_by_id = {
            w["id"]: (w.get("name") or w.get("displayName") or w["id"])
            for w in workspaces
        }
        if READ_WORKSPACE_IDS:
            allow = set(READ_WORKSPACE_IDS)
            ws_ids = [w for w in ws_ids if w in allow]
            print(f"  Applied allowlist -> {len(ws_ids)} workspace(s).")

        if not ws_ids:
            print("  No workspaces resolved. Check permissions (Fabric admin "
                  "role + 'SPs can call read-only admin APIs' tenant setting "
                  "if running as service principal).")
            return []

        # Quick visibility: show a few workspace names so the caller can sanity-
        # check that the tenant being scanned is the expected one.
        sample = workspaces[:5]
        print("  Sample workspaces:",
              ", ".join((w.get("name") or w.get("displayName") or w["id"])
                        for w in sample))

        # ---- Step 2: list notebooks ----------------------------------------
        notebooks = []

        if used_admin:
            # Tenant-wide first: one paged stream of every item in the tenant.
            # Filter notebooks (and optionally the workspace allowlist) client-
            # side. Much faster than N per-workspace calls.
            print("  Trying tenant-wide /v1/admin/items (single paged stream)...")
            try:
                tenant_nbs, first_sample = await _list_admin_items_tenant(session)
                print(f"  Tenant-wide returned {len(tenant_nbs)} notebooks.")
                if READ_WORKSPACE_IDS:
                    allow = set(READ_WORKSPACE_IDS)
                    tenant_nbs = [n for n in tenant_nbs
                                  if n["workspaceId"] in allow]
                    print(f"  After allowlist: {len(tenant_nbs)} notebooks.")
                if tenant_nbs:
                    for n in tenant_nbs:
                        n["workspaceName"] = ws_name_by_id.get(
                            n.get("workspaceId"), n.get("workspaceId", ""))
                    return tenant_nbs
                if first_sample:
                    print("  Tenant-wide saw items but none were notebooks. "
                          "Sample item shapes:")
                    import json as _json
                    for it in first_sample:
                        snippet = {k: it.get(k) for k in
                                   ("id", "type", "itemType", "displayName", "name",
                                    "workspaceId")}
                        print(f"    {_json.dumps(snippet)}")
                print("  Falling back to per-workspace /v1/admin/items ...")
            except Exception as e:
                print(f"  Tenant-wide FAILED: {e}; "
                      "falling back to per-workspace /v1/admin/items ...")

            sem = asyncio.Semaphore(50)

            async def one_admin(wid):
                async with sem:
                    return wid, await _list_admin_items_workspace(session, wid)

            results = await asyncio.gather(*(one_admin(w) for w in ws_ids))
        else:
            sem = asyncio.Semaphore(50)

            async def one_user(wid):
                async with sem:
                    return wid, await _list_user_items_workspace(session, wid)

            results = await asyncio.gather(*(one_user(w) for w in ws_ids))

        status_counts = {}
        first_nonempty_sample = None
        for wid, (items, status, sample) in results:
            status_counts[status] = status_counts.get(status, 0) + 1
            if items:
                notebooks.extend(items)
            elif sample and first_nonempty_sample is None:
                first_nonempty_sample = (wid, sample)

        print(f"  Per-workspace item listing status codes: {status_counts}")

        if not notebooks and first_nonempty_sample is not None:
            wid, sample = first_nonempty_sample
            print(f"\n  DIAGNOSTIC: workspace {wid} returned items but no "
                  f"items had type=='notebook'. Sample item shapes:")
            import json as _json
            for it in sample:
                snippet = {k: it.get(k) for k in
                           ("id", "type", "itemType", "displayName", "name")}
                print(f"    keys={list(it.keys())[:12]}")
                print(f"    {_json.dumps(snippet)}")
            print("  If 'type' uses a different label (e.g., 'SynapseNotebook'),")
            print("  add it to _item_to_notebook_row()'s match list.")

        # Stamp human-readable workspace name onto each notebook before
        # returning so it flows through the inventory + findings tables.
        for n in notebooks:
            n["workspaceName"] = ws_name_by_id.get(
                n.get("workspaceId"), n.get("workspaceId", ""))
        return notebooks


def _run_async(coro):
    """Run an asyncio coroutine from a Jupyter/Fabric notebook safely.

    The Fabric notebook kernel keeps its own event loop running, so calling
    loop.run_until_complete() on the main thread raises 'cannot run the event
    loop while another loop is running'. Running the coroutine on a worker
    thread gives it a fresh, isolated loop.
    """
    import threading

    try:
        asyncio.get_running_loop()
        running = True
    except RuntimeError:
        running = False

    if not running:
        return asyncio.run(coro)

    box = {}

    def worker():
        new_loop = asyncio.new_event_loop()
        try:
            asyncio.set_event_loop(new_loop)
            box["value"] = new_loop.run_until_complete(coro)
        except BaseException as e:
            box["error"] = e
        finally:
            new_loop.close()

    t = threading.Thread(target=worker, name="fabric-scanner-asyncio")
    t.start()
    t.join()
    if "error" in box:
        raise box["error"]
    return box["value"]


notebooks = [] if SOURCE_MODE == "lakehouse" \
                else _run_async(run_enumeration(_token))

if MAX_NOTEBOOKS and SOURCE_MODE == "api":
    notebooks = notebooks[:MAX_NOTEBOOKS]

# Build workspace lookup maps for cross-workspace detection. Each notebook
# entry already carries (workspaceId, workspaceName); derive bidirectional
# maps from that. In lakehouse mode the API list is empty, so seed it with
# the configured source workspace (if any).
_WS_NAME_BY_ID = {
    n["workspaceId"]: (n.get("workspaceName") or n["workspaceId"])
    for n in notebooks
    if n.get("workspaceId")
}
if SOURCE_MODE == "lakehouse" and SOURCE_LAKEHOUSE_WORKSPACE_ID:
    _WS_NAME_BY_ID[SOURCE_LAKEHOUSE_WORKSPACE_ID] = (
        SOURCE_LAKEHOUSE_WORKSPACE_NAME or SOURCE_LAKEHOUSE_WORKSPACE_ID
    )
_WS_NAME_BY_ID = {k.lower(): v for k, v in _WS_NAME_BY_ID.items()}
_WS_ID_BY_NAME = {v.lower(): k for k, v in _WS_NAME_BY_ID.items() if v}
print(f"  Workspace map: {len(_WS_NAME_BY_ID)} entries available "
      f"for cross-workspace lookup.")

if SOURCE_MODE == "lakehouse":
    print(f"[Lakehouse mode] Skipping API enumeration. Files will be "
          f"discovered by the next cell from: {source_uri}")
else:
    print(f"\nInventory: {len(notebooks)} notebooks total.")


In [ ]:
# === Build inventory DataFrame + partition ===================================
if SOURCE_MODE == "lakehouse":
    from pyspark.sql import functions as F
    file_df = (spark.read.format("binaryFile")
                 .option("recursiveFileLookup", "true")
                 .option("pathGlobFilter", SOURCE_FILE_GLOB)
                 .load(source_uri))
    n_total = file_df.count()
    if n_total == 0:
        raise SystemExit(
            f"No files matched glob '{SOURCE_FILE_GLOB}' under '{source_uri}'."
        )

    # Optional smoke-test cap.
    if MAX_NOTEBOOKS:
        file_df = file_df.limit(MAX_NOTEBOOKS)
        n_total = min(n_total, MAX_NOTEBOOKS)

    # Inventory snapshot (no content bytes) — written to the inventory table.
    snapshot = file_df.select(
        F.lit(SOURCE_LAKEHOUSE_WORKSPACE_ID or "").alias("workspace_id"),
        F.lit(SOURCE_LAKEHOUSE_WORKSPACE_NAME
              or SOURCE_LAKEHOUSE_WORKSPACE_ID
              or "").alias("workspace_name"),
        F.lit(SOURCE_LAKEHOUSE_ID or "").alias("source_lakehouse_id"),
        F.lit(SOURCE_LAKEHOUSE_NAME
              or SOURCE_LAKEHOUSE_ID
              or "").alias("source_lakehouse_name"),
        F.col("path").alias("notebook_id"),
        F.regexp_extract(F.col("path"), "([^/]+)$", 1).alias("display_name"),
    )

    if WRITE_TO_DEFAULT_LAKEHOUSE:
        snapshot.write.mode("overwrite").saveAsTable(inventory_target)
    else:
        (snapshot.write
            .format("delta")
            .mode("overwrite")
            .option("overwriteSchema", "true")
            .save(inventory_target))

    num_partitions = max(1, (n_total + TARGET_PARTITION_SIZE - 1)
                              // TARGET_PARTITION_SIZE)
    # inventory_df is the *scan input* — keep `path` + `content` columns.
    inventory_df = file_df.repartition(num_partitions)
    print(f"{n_total} files matched.  Scanning in {num_partitions} partitions.")
    print(f"Inventory persisted to: {inventory_target}")

else:
    if not notebooks:
        raise SystemExit("No notebooks found — nothing to scan.")

    inventory_rows = [
        Row(workspace_id=n["workspaceId"],
            workspace_name=(n.get("workspaceName") or n["workspaceId"]),
            source_lakehouse_id=None,
            source_lakehouse_name=None,
            notebook_id=n["id"],
            display_name=(n.get("displayName") or ""))
        for n in notebooks
    ]
    inventory_df = spark.createDataFrame(inventory_rows)

    if WRITE_TO_DEFAULT_LAKEHOUSE:
        inventory_df.write.mode("overwrite").saveAsTable(inventory_target)
    else:
        (inventory_df.write
            .format("delta")
            .mode("overwrite")
            .option("overwriteSchema", "true")
            .save(inventory_target))

    n_total = inventory_df.count()
    num_partitions = max(1, (n_total + TARGET_PARTITION_SIZE - 1)
                              // TARGET_PARTITION_SIZE)
    inventory_df = inventory_df.repartition(num_partitions)
    print(f"{n_total} notebooks across {num_partitions} partitions "
          f"(~{TARGET_PARTITION_SIZE} per partition).")
    print(f"Inventory persisted to: {inventory_target}")


In [ ]:
# === Fan out scanning with mapPartitions =====================================
# Strategy:
#   - The driver cell `04_scanner.py` already defines all detectors. Rather
#     than duplicating 600 lines of regex/AST helpers in each executor (the v1
#     pain point), we serialize the driver scanner source as a string and have
#     each partition function `exec()` it once to install all helpers locally.
#   - API mode: fetch each notebook async, scan the parsed JSON body.
#   - Lakehouse mode: scan the pre-loaded bytes directly.
#   - Both modes emit Row objects matching `result_schema` below.

# The driver scanner cell (`04_scanner.py`) is embedded as a raw string by the
# build script. Each partition function compiles it once into a fresh
# namespace, then calls scan_notebook_bytes().
_SCANNER_SRC = r'''# === Scanner engine: patterns, AST, entropy, URLs, cross-workspace ==========
# Driver-side reference. Each executor partition function inlines its own
# copy of these because Spark serialization across workers is most reliable
# when the partition function is self-contained.

SEVERITY_ORDER = ["low", "medium", "high", "critical"]
_MIN_SEV_IDX = SEVERITY_ORDER.index(MIN_SEVERITY.lower())


def _meets_severity(sev):
    try:
        return SEVERITY_ORDER.index(sev) >= _MIN_SEV_IDX
    except ValueError:
        return True


# --- URL schemes (NotebookAudit's wider list) -------------------------------
URL_PROTOCOL_SCHEMES = [
    "http://", "https://", "ftp://", "ftps://", "sftp://",
    "abfs://", "abfss://", "wasb://", "wasbs://", "adl://",
    "s3://", "s3a://", "s3n://", "gs://",
    "hdfs://", "viewfs://", "dbfs://",
    "jdbc:", "odbc:", "mongodb://", "mongodb+srv://",
    "postgresql://", "postgres://", "mysql://", "mssql://",
    "redis://", "rediss://", "amqp://", "amqps://",
    "kafka://", "kinesis://", "eventhubs://",
    "mqtt://", "mqtts://",
    "ssh://", "scp://", "smb://", "nfs://", "tcp://", "udp://",
]

_scheme_prefixes = []
for _s in URL_PROTOCOL_SCHEMES:
    _p = _s.rstrip(":/")
    if _p and _p not in _scheme_prefixes:
        _scheme_prefixes.append(re.escape(_p))
_schemes_alt = "|".join(sorted(_scheme_prefixes, key=len, reverse=True))

URL_RE = re.compile(
    r'(?P<url>(?:' + _schemes_alt + r')s?://[^\s"\'`<>)\]},;]+'
    r'|(?:' + _schemes_alt + r')s?:[^\s"\'`<>)\]},;]+'
    r'|\bwww\.[^\s"\'`<>)\]},;]+\.[^\s"\'`<>)\]},;]+)',
    re.IGNORECASE,
)
TRAILING_CHARS = '.,;:!?)]}>"\'`'

BENIGN_URL_PATTERNS = [
    re.compile(r'https?://(localhost|127\.0\.0\.1|0\.0\.0\.0)(:|/|$)', re.I),
    re.compile(r'https?://json\.schemastore\.org/', re.I),
    re.compile(r'https?://schemas?\.(microsoft|xmlsoap|w3)\.', re.I),
    re.compile(r'https?://(pypi|pip|conda|anaconda)\.(org|io)/', re.I),
    re.compile(r'https?://(docs|learn|aka)\.(microsoft|python|spark)\.', re.I),
    re.compile(r'https?://spark\.apache\.org/', re.I),
    re.compile(r'^file://', re.I),
]

READ_CONTEXT_RE = re.compile(
    r'(?:\brequests?\.(?:get|head|options)\s*\('
    r'|\bspark\.read\b'
    r'|\.read_\w+\s*\('
    r'|\bdownload\w*\s*\('
    r'|!\s*wget\b)',
    re.I,
)
WRITE_CONTEXT_RE = re.compile(
    r'(?:\brequests?\.(?:post|put|patch|delete)\s*\('
    r'|\.write\.\w+'
    r'|\.save\s*\('
    r'|\.to_\w+\s*\('
    r'|\bupload\w*\s*\('
    r'|\.write_\w+\s*\()',
    re.I,
)


# --- Suspicious imports (NotebookAudit) -------------------------------------
SUSPICIOUS_IMPORTS = {
    "requests", "httpx", "urllib", "urllib3", "http.client", "aiohttp", "pycurl",
    "boto3", "botocore", "azure.storage.blob", "azure.storage.filedatalake",
    "google.cloud.storage", "gcsfs", "s3fs", "adlfs",
    "paramiko", "ftplib", "pysftp", "smtplib",
    "socket", "websocket", "websockets",
    "grpc", "grpcio",
    "pymongo", "kafka", "confluent_kafka", "pika",
    "azure.eventhub", "azure.servicebus",
    "telegram", "slack_sdk", "discord", "twilio", "sendgrid",
    "msgraph", "O365", "office365",
    "ctypes", "cffi",
    "dropbox", "boxsdk", "pydrive", "pydrive2", "googleapiclient",
    "mega", "firebase_admin", "pyrebase", "supabase", "cloudinary", "b2sdk",
    "pyperclip", "clipboard",
}

IMPORT_CATEGORIES = {
    "requests": "http_exfiltration", "httpx": "http_exfiltration",
    "urllib": "http_exfiltration", "aiohttp": "http_exfiltration",
    "boto3": "cloud_storage_write", "botocore": "cloud_storage_write",
    "azure.storage.blob": "cloud_storage_write",
    "google.cloud.storage": "cloud_storage_write",
    "paramiko": "filesystem_exfiltration", "ftplib": "filesystem_exfiltration",
    "smtplib": "email_exfiltration",
    "socket": "raw_socket_network", "websocket": "raw_socket_network",
    "pymongo": "external_database_write", "kafka": "external_database_write",
    "telegram": "webhook_exfiltration", "slack_sdk": "webhook_exfiltration",
    "discord": "webhook_exfiltration",
    "msgraph": "api_exfiltration", "O365": "api_exfiltration",
    "dropbox": "cloud_storage_write", "boxsdk": "cloud_storage_write",
    "pydrive2": "cloud_storage_write", "googleapiclient": "cloud_storage_write",
    "mega": "cloud_storage_write", "firebase_admin": "cloud_storage_write",
    "supabase": "cloud_storage_write", "cloudinary": "cloud_storage_write",
    "ctypes": "native_code_execution", "cffi": "native_code_execution",
}


# --- 100+ regex patterns: union of NotebookAudit + v1's secret extensions ---
# Tuple format: (compiled_regex, category, severity, message)
PATTERNS = [
    # --- HTTP exfiltration ---
    (re.compile(r'(requests|httpx|urllib)\s*\.\s*(post|put|patch)\s*\(', re.I),
     "http_exfiltration", "high", "HTTP POST/PUT/PATCH request detected"),
    (re.compile(r'https?://[^\s\'"]+\.(ngrok|burpcollaborator|requestbin|pipedream|webhook\.site|hookbin|canarytokens|interactsh)', re.I),
     "http_exfiltration", "critical", "Known exfiltration service URL"),
    (re.compile(r'https?://\d{1,3}\.\d{1,3}\.\d{1,3}\.\d{1,3}[:/]', re.I),
     "http_exfiltration", "high", "HTTP request to raw IP address"),
    (re.compile(r'urllib\.request\.urlopen\s*\([^,)]+,\s*(data|b["\'])', re.I),
     "http_exfiltration", "high", "urllib.request.urlopen() with data — POST equivalent"),
    (re.compile(r'urllib\.request\.urlretrieve\s*\(|urlretrieve\s*\(', re.I),
     "http_exfiltration", "medium", "urlretrieve() downloads file to local disk"),

    # --- Cloud storage / file sharing services ---
    (re.compile(r'(api\.dropboxapi\.com|content\.dropboxapi\.com|dropbox\.com/oauth2)', re.I),
     "cloud_storage_write", "high", "Dropbox API endpoint"),
    (re.compile(r'\b(dropbox\.Dropbox|DropboxOAuth2FlowNoRedirect)\s*\(', re.I),
     "cloud_storage_write", "high", "Dropbox SDK client initialization"),
    (re.compile(r'(api\.box\.com|upload\.box\.com|box\.com/api)', re.I),
     "cloud_storage_write", "high", "Box.com API endpoint"),
    (re.compile(r'(drive\.google\.com|www\.googleapis\.com/upload/drive|www\.googleapis\.com/drive)', re.I),
     "cloud_storage_write", "high", "Google Drive API endpoint"),
    (re.compile(r'(mega\.nz|mega\.co\.nz|g\.api\.mega\.co\.nz)', re.I),
     "cloud_storage_write", "high", "MEGA cloud storage endpoint"),
    (re.compile(r'(wetransfer\.com/api|we\.tl)', re.I),
     "cloud_storage_write", "high", "WeTransfer API/URL"),
    (re.compile(r'(api\.pcloud\.com|pcloud\.com/oauth2)', re.I),
     "cloud_storage_write", "high", "pCloud API endpoint"),
    (re.compile(r'(firebaseio\.com|firebasestorage\.googleapis\.com)', re.I),
     "cloud_storage_write", "high", "Firebase endpoint"),
    (re.compile(r'\b\w+\.supabase\.co\b', re.I),
     "cloud_storage_write", "high", "Supabase endpoint"),
    (re.compile(r'(api\.cloudinary\.com|cloudinary\.uploader)', re.I),
     "cloud_storage_write", "high", "Cloudinary upload endpoint"),
    (re.compile(r'(backblazeb2\.com|b2_authorize_account|b2_upload_file)', re.I),
     "cloud_storage_write", "high", "Backblaze B2 storage endpoint"),
    (re.compile(r'(icloud\.com/api|setup\.icloud\.com)', re.I),
     "cloud_storage_write", "high", "iCloud API endpoint"),
    (re.compile(r'\b(upload_blob|upload_file|upload_fileobj|upload_from_string|upload_data|put_object|put_block_blob|put_item|put_record)\s*\(', re.I),
     "cloud_storage_write", "high", "Cloud storage upload SDK call"),

    # --- Paste / gist services ---
    (re.compile(r'(pastebin\.com/api|api\.paste\.ee|dpaste\.com/api|ix\.io|sprunge\.us|hastebin\.com)', re.I),
     "api_exfiltration", "high", "Paste service API — data exfiltration risk"),
    (re.compile(r'api\.github\.com/gists', re.I),
     "api_exfiltration", "high", "GitHub Gists API — data exfiltration risk"),
    (re.compile(r'(gitlab\.com/api.*snippets|snippets\.gitlab\.com)', re.I),
     "api_exfiltration", "high", "GitLab Snippets API"),
    (re.compile(r'api\.airtable\.com', re.I),
     "api_exfiltration", "high", "Airtable API endpoint"),
    (re.compile(r'api\.notion\.(com|so)', re.I),
     "api_exfiltration", "high", "Notion API endpoint"),

    # --- Webhooks ---
    (re.compile(r'https://hooks\.slack\.com/services/', re.I),
     "webhook_exfiltration", "critical", "Slack incoming webhook URL"),
    (re.compile(r'https://discord(app)?\.com/(api/)?webhooks/', re.I),
     "webhook_exfiltration", "critical", "Discord webhook URL"),
    (re.compile(r'https://api\.telegram\.org/bot', re.I),
     "webhook_exfiltration", "critical", "Telegram bot API"),
    (re.compile(r'https://[^\s]*\.execute-api\.[^\s]*\.amazonaws\.com', re.I),
     "webhook_exfiltration", "high", "AWS API Gateway endpoint"),
    (re.compile(r'https://[^\s]+(\.workers\.dev|\.vercel\.app|\.netlify\.app|\.pages\.dev|\.azurewebsites\.net)', re.I),
     "webhook_exfiltration", "high", "Serverless platform endpoint"),

    # --- Microsoft API abuse ---
    (re.compile(r'https://graph\.microsoft\.com/v\d+\.\d+/(me|users|groups|drives|sites)', re.I),
     "api_exfiltration", "high", "Microsoft Graph API call"),
    (re.compile(r'https://management\.azure\.com/subscriptions', re.I),
     "api_exfiltration", "high", "Azure Management API call"),
    (re.compile(r'https://api\.fabric\.microsoft\.com/v1/', re.I),
     "cross_workspace_operation", "high", "Fabric REST API call from within notebook"),
    (re.compile(r'https://api\.powerbi\.com/v1\.0/myorg/datasets/[^/]+/rows', re.I),
     "api_exfiltration", "critical", "Power BI streaming/push dataset API"),
    (re.compile(r'(GraphServiceClient|MSGraph|O365\.Account)\s*\(', re.I),
     "api_exfiltration", "critical", "Microsoft Graph SDK client"),

    # --- Credential access (NotebookAudit + v1's union) ---
    (re.compile(r'(mssparkutils|notebookutils)\.credentials\.(getSecret|getToken|getConnectionString|getFullConnectionString)', re.I),
     "credential_access", "high", "Fabric credential access"),
    (re.compile(r'(TokenLibrary|dbutils\.secrets)\.(get|getSecret|getToken)', re.I),
     "credential_access", "high", "Token/secret library access"),
    (re.compile(r'AccountKey\s*=\s*[A-Za-z0-9+/=]{40,}', re.I),
     "credential_access", "critical", "Hardcoded Storage Account Key"),
    (re.compile(r'(\?|&)sig=[A-Za-z0-9%+/=]{20,}', re.I),
     "credential_access", "critical", "SAS token with signature"),
    (re.compile(r'["\']https?://[^"\s]+\.blob\.core\.windows\.net[^"\s]*sig=[^"\s]+["\']', re.I),
     "credential_access", "critical", "SAS-tokened blob URL assignment"),
    (re.compile(r'(Password|pwd)\s*=\s*[^;\s"]{4,}', re.I),
     "credential_access", "high", "Hardcoded SQL password"),
    (re.compile(r'(client[_-]?secret|app[_-]?secret)\s*[=:]\s*["\']?[A-Za-z0-9~._-]{16,}', re.I),
     "credential_access", "critical", "Hardcoded client/app secret"),
    (re.compile(r'["\']Bearer\s+eyJ[A-Za-z0-9_-]{20,}', re.I),
     "credential_access", "critical", "Hardcoded Bearer JWT token"),
    (re.compile(r'SharedAccessKey\s*=\s*[A-Za-z0-9+/=]{30,}', re.I),
     "credential_access", "critical", "Hardcoded Event Hub / Service Bus key"),
    (re.compile(r'(openai[_-]?api[_-]?key|OPENAI_API_KEY)\s*[=:]\s*["\']?sk-[A-Za-z0-9]{20,}', re.I),
     "credential_access", "critical", "Hardcoded OpenAI API key"),
    (re.compile(r'(azure[_-]?openai[_-]?key|AZURE_OPENAI_KEY)\s*[=:]\s*["\']?[A-Fa-f0-9]{32}', re.I),
     "credential_access", "critical", "Hardcoded Azure OpenAI key"),
    (re.compile(r'(AccountKey|cosmosdb[_-]?key)\s*[=:]\s*["\']?[A-Za-z0-9+/=]{40,}', re.I),
     "credential_access", "critical", "Hardcoded Cosmos DB / Storage key"),
    (re.compile(r'(api[_-]?key|apikey|subscription[_-]?key|Ocp-Apim-Subscription-Key)\s*[=:]\s*["\']?[A-Za-z0-9]{16,}', re.I),
     "credential_access", "high", "Hardcoded API key"),
    (re.compile(r'(connection[_-]?string|conn[_-]?str)\s*[=:]\s*["\'][^"]{20,}["\']', re.I),
     "credential_access", "high", "Hardcoded connection string"),
    (re.compile(r'(secret|password|passwd|token|credential)\s*[=:]\s*["\'][^"\s]{8,}["\']', re.I),
     "credential_access", "medium", "Generic secret assignment"),
    (re.compile(r'spark\.conf\.set\s*\([^)]*account\.key[^)]*\)', re.I),
     "credential_access", "high", "Spark config with storage account key"),
    (re.compile(r'spark\.conf\.set\s*\(\s*["\'].*(account\.key|access\.key|secret|password|token)', re.I),
     "credential_access", "high", "Spark config with external credentials"),
    (re.compile(r'jdbc:[a-z]+://[^\s]*password=[^;\s"]{4,}', re.I),
     "credential_access", "critical", "JDBC URL with embedded password"),

    # --- Shell / package install ---
    (re.compile(r'^!pip\s+install\b|^%pip\s+install\b|^%conda\s+install\b', re.I | re.MULTILINE),
     "in_notebook_package_install", "high", "In-notebook package installation"),
    (re.compile(r'pip\s+install\s+.*--(index-url|extra-index-url)\s+http', re.I),
     "in_notebook_package_install", "critical", "pip install from non-PyPI index"),
    (re.compile(r'^!(curl|wget|nc|ncat|netcat|scp|rsync|ssh|sftp|ftp)\b', re.I | re.MULTILINE),
     "shell_execution", "critical", "Shell command for network file transfer"),
    (re.compile(r'(os\.system|os\.popen|subprocess\.(run|Popen|call|check_output))\s*\(', re.I),
     "shell_execution", "high", "Shell command execution"),

    # --- Spark external writes ---
    (re.compile(r'\.format\s*\(\s*["\'](?:com\.mongodb|org\.apache\.spark\.sql\.cassandra|kafka|es\.spark)', re.I),
     "spark_external_write", "high", "Spark write with external connector format"),
    (re.compile(r'writeStream.*\.format\s*\(\s*["\'](?:kafka|eventhubs|socket|mqtt|kinesis)["\']\s*\)', re.I),
     "spark_streaming_exfiltration", "critical", "Spark streaming to external broker"),
    (re.compile(r'writeStream.*foreachBatch', re.I),
     "spark_streaming_exfiltration", "high", "Spark streaming foreachBatch — custom sink"),
    (re.compile(r'(dbutils\.fs\.cp|mssparkutils\.fs\.cp|shutil\.copy).*(?:s3|wasb|gs|abfss|adl|ftp)', re.I),
     "filesystem_exfiltration", "critical", "File copy to external cloud storage"),
    (re.compile(r'(dbutils\.fs\.mount|mssparkutils\.fs\.mount)', re.I),
     "spark_external_write", "high", "Mounting external storage"),
    (re.compile(r'(mssparkutils|notebookutils)\.fs\.(put|append|mv)\s*\(', re.I),
     "filesystem_exfiltration", "high", "mssparkutils/notebookutils file write/move"),

    # --- JDBC / database ---
    (re.compile(r'jdbc:(mysql|postgresql|sqlserver|oracle|db2|redshift|snowflake|bigquery)://[^\s\'"]+', re.I),
     "external_database_write", "high", "JDBC connection string to external database"),
    (re.compile(r'Server\s*=\s*[^;]+\.database\.windows\.net', re.I),
     "external_database_write", "high", "Azure SQL connection string"),

    # --- Email ---
    (re.compile(r'(smtplib\.SMTP|MIMEMultipart|MIMEText|send_message|sendmail)\s*[(\.]', re.I),
     "email_exfiltration", "high", "Email sending capability"),
    (re.compile(r'(api\.sendgrid\.com|api\.mailgun\.net|mandrillapp\.com/api|api\.sparkpost\.com)', re.I),
     "email_exfiltration", "high", "Transactional email API endpoint"),

    # --- DNS exfiltration ---
    (re.compile(r'(dns\.resolver|dnspython|socket\.getaddrinfo)', re.I),
     "dns_exfiltration", "high", "DNS resolution — potential DNS exfiltration"),

    # --- Encoding / obfuscation ---
    (re.compile(r'(exec|eval)\s*\(\s*(base64|codecs|bytes\.fromhex|bytearray)', re.I),
     "dynamic_code_execution", "critical", "Execution of decoded/deobfuscated code"),
    (re.compile(r'getattr\s*\(\s*__builtins__', re.I),
     "dynamic_code_execution", "critical", "Dynamic access to builtins — obfuscation"),

    # --- Polars / Pandas writes ---
    (re.compile(r'\.write_(?:delta|parquet|csv|json|ndjson|ipc|arrow|avro|excel|database)\s*\(', re.I),
     "filesystem_exfiltration", "medium", "Polars DataFrame write to file/database"),
    (re.compile(r'\.to_(?:csv|json|parquet|sql|excel|html|hdf|pickle|orc|feather|stata|gbq|xml)\s*\(', re.I),
     "filesystem_exfiltration", "medium", "Pandas DataFrame export to file/database"),

    # --- SQL DML ---
    (re.compile(r'\.execute\s*\(\s*[f"\']*(INSERT\s+INTO|UPDATE\s+|DELETE\s+FROM|MERGE\s+INTO)', re.I),
     "external_database_write", "medium", "SQL DML via .execute()"),
    (re.compile(r'spark\.sql\s*\(\s*f?["\']*(INSERT\s+INTO|UPDATE\s+|DELETE\s+FROM|MERGE\s+INTO)', re.I),
     "spark_external_write", "medium", "spark.sql() with data-modifying statement"),

    # --- Fabric APIs that need migration / update review ---
    # fastcp is Fabric's high-throughput bulk file copy utility. Any notebook
    # using it deserves a migration / review pass: the source/destination URIs
    # are usually OneLake or external storage, and the API surface has shifted
    # from mssparkutils.fs.fastcp -> notebookutils.fs.fastcp in Fabric.
    (re.compile(r'\b(?:mssparkutils|notebookutils)\.fs\.fastcp\s*\(', re.I),
     "needs_update", "medium",
     "fastcp() bulk copy detected - flag for migration / review"),
    (re.compile(r'(?<!\.)\bfastcp\s*\(', re.I),
     "needs_update", "medium",
     "fastcp() bulk copy detected (bare call) - flag for migration / review"),
    (re.compile(r'from\s+(?:mssparkutils|notebookutils)\.fs\s+import\s+[^\n#]*\bfastcp\b', re.I),
     "needs_update", "medium",
     "fastcp imported - flag for migration / review"),
]


# --- Potential-write (v1): destination-agnostic write API detection. -------
# Catches notebooks that build the destination at runtime (the URL never
# appears as a literal). Surfaced as severity=medium category=potential_write.
WRITE_CALL_RE = re.compile(
    r"\.write_(?:delta|parquet|csv|json|ndjson|ipc|arrow|avro|excel|database)\s*\("
    r"|\.to_(?:csv|json|parquet|sql|excel|html|hdf|pickle|orc|feather|stata|gbq|xml)\s*\("
    r"|\.write\.(?:format|mode|option|partitionBy|saveAsTable|save|insertInto|jdbc)\s*\("
    r"|\.writeStream\.(?:format|trigger|option|outputMode|start|toTable|foreachBatch)\s*\("
    r"|\brequests?\.(?:post|put|patch|delete)\s*\("
    r"|\bhttpx?\.(?:post|put|patch|delete)\s*\("
    r"|\b(?:upload_blob|upload_file|upload_fileobj|upload_from_string|"
    r"upload_data|put_object|put_block_blob|put_item|put_record)\s*\("
    r"|\bmssparkutils\.fs\.(?:put|append|mv|cp|fastcp)\s*\("
    r"|\bnotebookutils\.fs\.(?:put|append|mv|cp|fastcp)\s*\("
    r"|\bdbutils\.fs\.(?:put|cp|mv)\s*\("
    r"|\.execute\s*\(\s*[fbur]*['\"]?(?:INSERT|UPDATE|DELETE|MERGE)\s"
    r"|\bspark\.sql\s*\(\s*f?['\"]?(?:INSERT|UPDATE|DELETE|MERGE)\s"
    r"|\bopen\s*\([^,)]*,\s*['\"][wax]b?",
    re.I,
)


# --- Cross-workspace destination parsing (v1) ------------------------------
WORKSPACE_URL_RE = re.compile(
    r"(?:"
    r"abfss?://([^@/\s]+)@onelake\.dfs\.fabric\.microsoft\.com"
    r"|https?://[^/\s]*\.fabric\.microsoft\.com/(?:[^?\s]*?/)?(?:v1/)?workspaces/([0-9a-f-]{8,})"
    r"|https?://(?:app|api)\.powerbi\.com/(?:[^/\s]*/)*?groups/([0-9a-f-]{8,})"
    r")",
    re.IGNORECASE,
)
GUID_RE = re.compile(
    r"^[0-9a-f]{8}-?[0-9a-f]{4}-?[0-9a-f]{4}-?[0-9a-f]{4}-?[0-9a-f]{12}$",
    re.IGNORECASE,
)


def parse_dest_workspace(url):
    if not url:
        return None
    m = WORKSPACE_URL_RE.search(url)
    if not m:
        return None
    return m.group(1) or m.group(2) or m.group(3)


def resolve_dest_workspace(url, src_ws_id, src_ws_name,
                           ws_name_by_id=None, ws_id_by_name=None):
    raw = parse_dest_workspace(url)
    if not raw:
        return None, None, None
    name_map = ws_name_by_id or {}
    id_map = ws_id_by_name or {}
    if GUID_RE.match(raw):
        dest_id = raw.lower()
        dest_name = name_map.get(dest_id, "") or name_map.get(raw, "")
    else:
        dest_name = raw
        dest_id = id_map.get(raw.lower(), "")
    src_id_l = (src_ws_id or "").lower()
    src_name_l = (src_ws_name or "").lower()
    if dest_id and src_id_l and GUID_RE.match(src_id_l or ""):
        cross = (dest_id != src_id_l)
    elif dest_name and src_name_l:
        cross = (dest_name.lower() != src_name_l)
    else:
        cross = None
    return (dest_id or None), (dest_name or None), cross


# --- Entropy analysis (NotebookAudit) --------------------------------------
def _shannon_entropy(s):
    if not s:
        return 0.0
    freq = {}
    for c in s:
        freq[c] = freq.get(c, 0) + 1
    n = len(s)
    return -sum((count / n) * math.log2(count / n) for count in freq.values())


# --- Snippet redaction for credentials (v1) --------------------------------
def _redact(s, keep=4):
    if not s:
        return s
    s = s.strip()
    if len(s) <= keep * 2 + 3:
        return "***"
    return s[:keep] + "*" * (len(s) - keep * 2) + s[-keep:]


def _trim_snippet(text, max_bytes):
    t = (text or "").strip()
    if len(t) <= max_bytes:
        return t
    return t[: max_bytes - 1] + "…"


# --- AST: suspicious imports + eval/exec (NotebookAudit) -------------------
def _ast_scan(code_str):
    findings = []
    try:
        tree = ast.parse(code_str)
    except SyntaxError:
        return findings
    for node in ast.walk(tree):
        if isinstance(node, (ast.Import, ast.ImportFrom)):
            mods = ([a.name for a in node.names] if isinstance(node, ast.Import)
                    else ([node.module] if node.module else []))
            for mod in mods:
                if not mod:
                    continue
                top = mod.split(".")[0]
                matched = mod if mod in SUSPICIOUS_IMPORTS else (
                    top if top in SUSPICIOUS_IMPORTS else None)
                if matched:
                    cat = IMPORT_CATEGORIES.get(matched, "suspicious_import")
                    findings.append({
                        "finding_type": "import",
                        "category": cat, "severity": "high",
                        "message": f"Import of suspicious module: {mod}",
                        "line": getattr(node, "lineno", 0),
                    })
        if isinstance(node, ast.Call):
            if isinstance(node.func, ast.Name) and node.func.id in ("eval", "exec", "compile"):
                findings.append({
                    "finding_type": "dynamic_exec",
                    "category": "dynamic_code_execution", "severity": "critical",
                    "message": f"{node.func.id}() call — dynamic code execution",
                    "line": node.lineno,
                })
    return findings


# --- Multi-format code extractor (NotebookAudit, extended for markdown) ---
def _extract_blocks(content_bytes, file_path, include_md_and_outputs=True):
    """Yields list of (text, cell_index, source_kind, part_path).

    source_kind ∈ {"code", "markdown", "output", "raw"}.
    part_path is "" for plain .ipynb / .py inputs; for Fabric Item JSON it is
    the path of the part inside `definition.parts` (e.g. "notebook-content.py").
    """
    try:
        text = content_bytes.decode("utf-8", errors="ignore") if isinstance(
            content_bytes, (bytes, bytearray)) else str(content_bytes)
    except Exception:
        return []

    fp = (file_path or "").lower()
    if fp.endswith(".py"):
        return [(text, 0, "code", "")]
    if fp.endswith(".md"):
        return [(text, 0, "markdown", "")]

    if fp.endswith(".ipynb") or fp.endswith(".json") or text.lstrip().startswith("{"):
        try:
            data = json.loads(text)
        except (json.JSONDecodeError, ValueError):
            return [(text, 0, "raw", "")]
        if not isinstance(data, dict):
            return [(text, 0, "raw", "")]

        blocks = []

        # Standard .ipynb
        if "cells" in data and isinstance(data["cells"], list):
            for i, cell in enumerate(data["cells"]):
                ct = cell.get("cell_type")
                src = cell.get("source", [])
                txt = "".join(src) if isinstance(src, list) else str(src)
                if ct == "code":
                    blocks.append((txt, i, "code", ""))
                elif ct == "markdown" and include_md_and_outputs:
                    blocks.append((txt, i, "markdown", ""))
                if include_md_and_outputs:
                    for out in cell.get("outputs", []) or []:
                        if not isinstance(out, dict):
                            continue
                        t = out.get("text", "")
                        if t:
                            ot = "".join(t) if isinstance(t, list) else str(t)
                            blocks.append((ot, i, "output", ""))
                        # Stream / display_data text/plain
                        data_field = out.get("data") or {}
                        for key, val in data_field.items():
                            if "text" in key.lower():
                                vt = "".join(val) if isinstance(val, list) else str(val)
                                blocks.append((vt, i, "output", ""))
            return blocks

        # Fabric Item JSON: definition.parts (base64-encoded payloads)
        parts = (data.get("definition") or {}).get("parts") or []
        if parts:
            for i, part in enumerate(parts):
                path = part.get("path", "") or ""
                payload = part.get("payload", "") or ""
                if not payload:
                    continue
                try:
                    decoded = base64.b64decode(payload).decode("utf-8", errors="ignore")
                except Exception:
                    continue
                if path.endswith(".ipynb"):
                    try:
                        sub = json.loads(decoded)
                    except (json.JSONDecodeError, ValueError):
                        blocks.append((decoded, i, "raw", path))
                        continue
                    for j, cell in enumerate(sub.get("cells", []) or []):
                        ct = cell.get("cell_type")
                        src = cell.get("source", [])
                        txt = "".join(src) if isinstance(src, list) else str(src)
                        if ct == "code":
                            blocks.append((txt, j, "code", path))
                        elif ct == "markdown" and include_md_and_outputs:
                            blocks.append((txt, j, "markdown", path))
                        if include_md_and_outputs:
                            for out in cell.get("outputs", []) or []:
                                if not isinstance(out, dict):
                                    continue
                                t = out.get("text", "")
                                if t:
                                    ot = "".join(t) if isinstance(t, list) else str(t)
                                    blocks.append((ot, j, "output", path))
                elif path.endswith(".py"):
                    blocks.append((decoded, i, "code", path))
                elif path.endswith(".md"):
                    if include_md_and_outputs:
                        blocks.append((decoded, i, "markdown", path))
                else:
                    if include_md_and_outputs:
                        blocks.append((decoded, i, "raw", path))
            if blocks:
                return blocks

        # Fabric Item JSON: properties.cells
        cells = (data.get("properties") or {}).get("cells") or []
        if cells:
            for i, cell in enumerate(cells):
                ct = cell.get("cell_type")
                src = cell.get("source", [])
                txt = "".join(src) if isinstance(src, list) else str(src)
                if ct == "code":
                    blocks.append((txt, i, "code", ""))
                elif ct == "markdown" and include_md_and_outputs:
                    blocks.append((txt, i, "markdown", ""))
            return blocks

    return [(text, 0, "raw", "")]


def _scan_one_block(text, cell_idx, source_kind, code_cells_only_for_pattern=True):
    """Run all detectors on one block. Returns a list of finding dicts."""
    findings = []
    if not text or not text.strip():
        return findings
    lines = text.splitlines()

    is_code = (source_kind == "code")

    # 1. Regex PATTERNS (only on code cells)
    if is_code:
        for pattern, category, severity, message in PATTERNS:
            for m in pattern.finditer(text):
                if not _meets_severity(severity):
                    continue
                line_no = text[: m.start()].count("\n") + 1
                snippet_line = (lines[line_no - 1] if line_no <= len(lines) else "").strip()
                snippet = _trim_snippet(snippet_line, MAX_SNIPPET_BYTES)
                if category == "credential_access":
                    snippet = _redact(snippet, keep=4)
                findings.append({
                    "finding_type": "pattern",
                    "category": category, "severity": severity,
                    "message": message, "line": line_no,
                    "code_snippet": snippet,
                    "url": None, "direction": None,
                })

    # 2. AST (only on code cells)
    if is_code:
        for f in _ast_scan(text):
            if _meets_severity(f["severity"]):
                line = f.get("line", 0)
                snippet_line = (lines[line - 1] if 0 < line <= len(lines) else "").strip()
                f["code_snippet"] = _trim_snippet(snippet_line, MAX_SNIPPET_BYTES)
                f["url"] = None
                f["direction"] = None
                findings.append(f)

    # 3. URL detection (code, markdown, outputs)
    seen_urls = set()
    url_severity = "medium" if is_code else "low"
    if _meets_severity(url_severity):
        for m in URL_RE.finditer(text):
            url = m.group("url")
            while url and url[-1] in TRAILING_CHARS:
                url = url[:-1]
            if not url:
                continue
            ul = url.lower().rstrip("/")
            if ul in seen_urls:
                continue
            if any(bp.search(url) for bp in BENIGN_URL_PATTERNS):
                continue
            seen_urls.add(ul)
            if is_code:
                ctx_start = max(0, m.start() - 300)
                ctx = text[ctx_start: m.start()]
                has_read = READ_CONTEXT_RE.search(ctx)
                has_write = WRITE_CONTEXT_RE.search(ctx)
                if has_read and has_write:
                    direction = "read_write"
                elif has_write:
                    direction = "write"
                elif has_read:
                    direction = "read"
                else:
                    direction = "unknown"
            else:
                direction = "reference"
            line_no = text[: m.start()].count("\n") + 1
            snippet_line = (lines[line_no - 1] if line_no <= len(lines) else "").strip()
            findings.append({
                "finding_type": "url",
                "category": "external_url_reference",
                "severity": url_severity,
                "message": f"External URL/URI [direction: {direction}]: {url[:100]}",
                "line": line_no,
                "code_snippet": _trim_snippet(snippet_line, MAX_SNIPPET_BYTES),
                "url": url, "direction": direction,
            })

    # 4. Entropy (only on code cells)
    if is_code and _meets_severity("high"):
        try:
            tree = ast.parse(text)
            for node in ast.walk(tree):
                if isinstance(node, ast.Constant) and isinstance(node.value, str):
                    s = node.value
                    if len(s) > 40:
                        ent = _shannon_entropy(s)
                        if ent > 4.5 and not s.lstrip().upper().startswith(
                                ("SELECT", "INSERT", "CREATE", "ALTER", "UPDATE", "DELETE", "MERGE", "WITH")):
                            line = getattr(node, "lineno", 0)
                            findings.append({
                                "finding_type": "entropy",
                                "category": "encoding_obfuscation",
                                "severity": "high",
                                "message": f"High-entropy string literal (entropy={ent:.2f}, len={len(s)})",
                                "line": line,
                                "code_snippet": _trim_snippet(s, MAX_SNIPPET_BYTES),
                                "url": None, "direction": None,
                            })
        except SyntaxError:
            pass

    # 5. Potential-write (v1): destination-agnostic write call detection.
    if is_code and _meets_severity("medium"):
        seen_pw = set()
        for m in WRITE_CALL_RE.finditer(text):
            sig = m.group(0)
            line_no = text[: m.start()].count("\n") + 1
            snippet_line = (lines[line_no - 1] if line_no <= len(lines) else "").strip()
            key = (line_no, sig.strip())
            if key in seen_pw:
                continue
            seen_pw.add(key)
            findings.append({
                "finding_type": "potential_write",
                "category": "potential_write",
                "severity": "medium",
                "message": f"Write-API call detected: {sig.strip()[:60]}",
                "line": line_no,
                "code_snippet": _trim_snippet(snippet_line, MAX_SNIPPET_BYTES),
                "url": None, "direction": "potential_write",
            })

    return findings


def _extract_attached_lakehouse(content_bytes, file_label):
    """Inspects the notebook's metadata to find its default ("attached")
    Lakehouse. Looks at metadata.dependencies.lakehouse — the standard
    Fabric/Synapse notebook attachment block.

    Returns a dict with keys:
        attached_lakehouse_id
        attached_lakehouse_name
        attached_lakehouse_workspace_id
        attached_lakehouse_workspace_name  (always None here; resolved by the
                                            executor via the broadcast ws map)
    Any field that cannot be determined is None.
    Handles plain .ipynb, Fabric Item JSON with definition.parts (base64
    .ipynb payloads), and Fabric Item JSON with properties.metadata.
    """
    empty = {
        "attached_lakehouse_id": None,
        "attached_lakehouse_name": None,
        "attached_lakehouse_workspace_id": None,
        "attached_lakehouse_workspace_name": None,
    }
    try:
        text = content_bytes.decode("utf-8", errors="ignore") if isinstance(
            content_bytes, (bytes, bytearray)) else str(content_bytes)
    except Exception:
        return dict(empty)

    fp = (file_label or "").lower()
    if fp.endswith(".py") or fp.endswith(".md"):
        return dict(empty)

    def _from_meta(meta):
        if not isinstance(meta, dict):
            return None
        deps = meta.get("dependencies")
        if not isinstance(deps, dict):
            return None
        lh = deps.get("lakehouse")
        if not isinstance(lh, dict):
            return None
        lh_id = lh.get("default_lakehouse") or lh.get("default_lakehouse_id")
        lh_name = lh.get("default_lakehouse_name")
        lh_wsid = lh.get("default_lakehouse_workspace_id")
        # Some exports nest known_lakehouses[0] when default_* is missing.
        if not lh_id:
            known = lh.get("known_lakehouses") or []
            if isinstance(known, list) and known and isinstance(known[0], dict):
                lh_id = known[0].get("id") or known[0].get("lakehouse_id")
        if not (lh_id or lh_name or lh_wsid):
            return None
        return {
            "attached_lakehouse_id": lh_id or None,
            "attached_lakehouse_name": lh_name or None,
            "attached_lakehouse_workspace_id": lh_wsid or None,
            "attached_lakehouse_workspace_name": None,
        }

    try:
        data = json.loads(text)
    except (json.JSONDecodeError, ValueError):
        return dict(empty)
    if not isinstance(data, dict):
        return dict(empty)

    # Plain .ipynb
    result = _from_meta(data.get("metadata"))
    if result:
        return result

    # Fabric Item JSON: definition.parts (base64-encoded .ipynb payloads)
    parts = (data.get("definition") or {}).get("parts") or []
    for part in parts:
        path = part.get("path", "") or ""
        payload = part.get("payload", "") or ""
        if not payload or not path.endswith(".ipynb"):
            continue
        try:
            decoded = base64.b64decode(payload).decode("utf-8", errors="ignore")
            sub = json.loads(decoded)
        except Exception:
            continue
        result = _from_meta(sub.get("metadata"))
        if result:
            return result

    # Fabric Item JSON: properties.metadata
    props = data.get("properties")
    if isinstance(props, dict):
        result = _from_meta(props.get("metadata"))
        if result:
            return result

    return dict(empty)


def scan_notebook_bytes(content_bytes, file_label,
                        include_md_and_outputs=True):
    """Top-level driver scan. Returns list of finding dicts, each annotated
    with cell_index, source_kind, part_path, and attached_lakehouse_*.
    URL findings carry no dest_workspace yet (the executor adds that). Same
    for attached_lakehouse_workspace_name (executor resolves via broadcast)."""
    out = []
    lh = _extract_attached_lakehouse(content_bytes, file_label)
    for text, cell_idx, source_kind, part_path in _extract_blocks(
            content_bytes, file_label, include_md_and_outputs):
        for f in _scan_one_block(text, cell_idx, source_kind):
            f["cell_index"] = int(cell_idx)
            f["source_kind"] = source_kind
            f["part_path"] = part_path or ""
            f.update(lh)
            out.append(f)
    return out


print(f"Loaded {len(PATTERNS)} regex patterns "
      f"across {len(set(p[1] for p in PATTERNS))} categories.")
print(f"Suspicious-import list: {len(SUSPICIOUS_IMPORTS)} entries.")
print(f"URL schemes detected: {len(URL_PROTOCOL_SCHEMES)}.")
'''


_FABRIC_BASE_BC = FABRIC_BASE
_CONC_BC = EXECUTOR_CONCURRENCY
_WS_NAME_BY_ID_BC = _WS_NAME_BY_ID
_WS_ID_BY_NAME_BC = _WS_ID_BY_NAME
_MAX_RETRIES_BC = MAX_RETRIES
_AUDIENCE_BC = TOKEN_AUDIENCE
_FALLBACK_TOKEN_BC = _token if SOURCE_MODE == "api" else None
_MIN_SEV_BC = MIN_SEVERITY
_MAX_SNIP_BC = MAX_SNIPPET_BYTES
_SCAN_MD_BC = SCAN_MARKDOWN_AND_OUTPUTS

# Source-LH constants — populated by cell 2 either from explicit config or
# from notebookutils.runtime.context auto-detection. Broadcast to executors
# so every finding row carries the source-LH provenance (workspace + LH).
# In API mode these are not used (workspace info comes from the API list).
_LH_WS_BC  = SOURCE_LAKEHOUSE_WORKSPACE_ID or ""
_LH_WSN_BC = (SOURCE_LAKEHOUSE_WORKSPACE_NAME
              or SOURCE_LAKEHOUSE_WORKSPACE_ID
              or "")
_LH_ID_BC  = SOURCE_LAKEHOUSE_ID or ""
_LH_LHN_BC = (SOURCE_LAKEHOUSE_NAME
              or SOURCE_LAKEHOUSE_ID
              or "")


def _build_scanner_ns():
    """Compile the embedded driver-scanner source into a fresh namespace.
    Called once per partition function invocation."""
    import re as _re, ast as _ast, math as _math, base64 as _b64, json as _json
    ns = {
        "re": _re, "ast": _ast, "math": _math, "base64": _b64, "json": _json,
        "MIN_SEVERITY": _MIN_SEV_BC,
        "MAX_SNIPPET_BYTES": _MAX_SNIP_BC,
        "__builtins__": __builtins__,
    }
    exec(_SCANNER_SRC, ns)
    return ns


def _row_from_finding(ns, f, wid, wname, nb_id, nb_name, scanned_at,
                      lh_src_id=None, lh_src_name=None):
    """Build a result Row from one finding dict, resolving dest_workspace
    and the attached-lakehouse workspace name from the broadcast map.
    `lh_src_id` / `lh_src_name` identify the lakehouse the file was READ FROM
    (constant for the whole partition in lakehouse mode, None in API mode)."""
    from pyspark.sql import Row
    url = f.get("url")
    direction = f.get("direction")
    d_id, d_name, cross = ns["resolve_dest_workspace"](
        url or "", wid or "", wname or "",
        _WS_NAME_BY_ID_BC or {}, _WS_ID_BY_NAME_BC or {},
    ) if url else (None, None, None)
    # Resolve attached-lakehouse workspace name from the broadcast map when
    # the notebook metadata only provided the workspace_id.
    lh_wsid = f.get("attached_lakehouse_workspace_id")
    lh_wsname = f.get("attached_lakehouse_workspace_name")
    if lh_wsid and not lh_wsname and (_WS_NAME_BY_ID_BC or {}):
        lh_wsname = (_WS_NAME_BY_ID_BC or {}).get(lh_wsid.lower())
    return Row(
        workspace_id=wid, workspace_name=wname,
        source_lakehouse_id=lh_src_id, source_lakehouse_name=lh_src_name,
        notebook_id=nb_id, display_name=nb_name,
        attached_lakehouse_id=f.get("attached_lakehouse_id"),
        attached_lakehouse_name=f.get("attached_lakehouse_name"),
        attached_lakehouse_workspace_id=lh_wsid,
        attached_lakehouse_workspace_name=lh_wsname,
        part_path=f.get("part_path") or None,
        source_kind=f.get("source_kind") or None,
        cell_index=int(f.get("cell_index", 0)),
        line_number=int(f.get("line", 0)),
        finding_type=f.get("finding_type"),
        category=f.get("category"),
        severity=f.get("severity"),
        message=f.get("message"),
        code_snippet=f.get("code_snippet"),
        url=url, direction=direction,
        dest_workspace_id=d_id, dest_workspace_name=d_name,
        cross_workspace=cross,
        error=None,
        scanned_at=scanned_at,
    )


def _empty_row(wid, wname, nb_id, nb_name, scanned_at, err=None,
               lh_id=None, lh_name=None, lh_wsid=None, lh_wsname=None,
               lh_src_id=None, lh_src_name=None):
    """Inventory row for notebooks with no findings (or an error). Still
    carries the attached-lakehouse columns so the inventory join shows them.
    Source-LH columns are carried through too so empty-result files still
    have provenance."""
    from pyspark.sql import Row
    return Row(
        workspace_id=wid, workspace_name=wname,
        source_lakehouse_id=lh_src_id, source_lakehouse_name=lh_src_name,
        notebook_id=nb_id, display_name=nb_name,
        attached_lakehouse_id=lh_id,
        attached_lakehouse_name=lh_name,
        attached_lakehouse_workspace_id=lh_wsid,
        attached_lakehouse_workspace_name=lh_wsname,
        part_path=None, source_kind=None,
        cell_index=0, line_number=0,
        finding_type=None, category=None, severity=None,
        message=None, code_snippet=None,
        url=None, direction=None,
        dest_workspace_id=None, dest_workspace_name=None,
        cross_workspace=None,
        error=err, scanned_at=scanned_at,
    )


# --- API-mode partition: async-fetch each notebook, scan the bytes ----------
def fetch_partition(rows):
    import asyncio, aiohttp, json
    from datetime import datetime, timezone

    rows = list(rows)
    if not rows:
        return iter([])

    # Refresh token on executor for long-running jobs.
    try:
        import notebookutils
        token = notebookutils.credentials.getToken(_AUDIENCE_BC)
    except Exception:
        try:
            import mssparkutils
            token = mssparkutils.credentials.getToken(_AUDIENCE_BC)
        except Exception:
            token = _FALLBACK_TOKEN_BC

    ns = _build_scanner_ns()
    _scan_bytes = ns["scan_notebook_bytes"]
    _extract_lh = ns["_extract_attached_lakehouse"]
    _ws_map = _WS_NAME_BY_ID_BC or {}

    headers = {"Authorization": f"Bearer {token}"}
    sem = asyncio.Semaphore(_CONC_BC)

    async def fetch_one(session, wid, wname, iid, name):
        url = (f"{_FABRIC_BASE_BC}/v1/workspaces/{wid}/items/{iid}"
               f"/getDefinition?format=ipynb")
        async with sem:
            for attempt in range(_MAX_RETRIES_BC + 1):
                try:
                    async with session.post(url) as r:
                        if r.status == 200:
                            return wid, wname, iid, name, await r.json(), None
                        if r.status == 202:
                            poll_url = r.headers.get("Location")
                            if not poll_url:
                                return wid, wname, iid, name, None, "202 without Location"
                        elif r.status in (429, 500, 502, 503, 504):
                            await asyncio.sleep(
                                float(r.headers.get("Retry-After",
                                                    str(min(2 ** attempt, 60))))
                            )
                            continue
                        else:
                            txt = await r.text()
                            return wid, wname, iid, name, None, f"HTTP {r.status}: {txt[:200]}"
                    for _ in range(180):
                        await asyncio.sleep(2)
                        async with session.get(poll_url) as p:
                            if p.status == 200:
                                pj = await p.json()
                                if pj.get("status") == "Succeeded":
                                    async with session.get(poll_url + "/result") as res:
                                        if res.status == 200:
                                            return wid, wname, iid, name, await res.json(), None
                                        else:
                                            return wid, wname, iid, name, None, f"result HTTP {res.status}"
                                if pj.get("status") == "Failed":
                                    return wid, wname, iid, name, None, f"LRO failed: {pj}"
                            elif p.status == 429:
                                await asyncio.sleep(float(p.headers.get("Retry-After", "5")))
                    return wid, wname, iid, name, None, "LRO poll timeout"
                except Exception as e:
                    if attempt == _MAX_RETRIES_BC:
                        return wid, wname, iid, name, None, f"{type(e).__name__}: {e}"
                    await asyncio.sleep(min(2 ** attempt, 30))

    async def run_all():
        connector = aiohttp.TCPConnector(limit=_CONC_BC * 2)
        timeout = aiohttp.ClientTimeout(total=1200, connect=30)
        async with aiohttp.ClientSession(connector=connector,
                                         headers=headers,
                                         timeout=timeout) as session:
            tasks = [fetch_one(session, r.workspace_id, r.workspace_name,
                               r.notebook_id, r.display_name) for r in rows]
            return await asyncio.gather(*tasks, return_exceptions=False)

    loop = asyncio.new_event_loop()
    try:
        results = loop.run_until_complete(run_all())
    finally:
        loop.close()

    now = datetime.now(timezone.utc)
    out_rows = []
    for wid, wname, iid, name, body, err in results:
        if err is not None or body is None:
            # Best-effort: we couldn't fetch the notebook, so we don't know its
            # attached lakehouse either. Still emit a row for inventory join.
            out_rows.append(_empty_row(wid, wname, iid, name, now, err))
            continue
        try:
            body_bytes = json.dumps(body).encode("utf-8")
            label = (name or iid or "") + ".ipynb"
            lh = _extract_lh(body_bytes, label)
            if lh.get("attached_lakehouse_workspace_id") and not lh.get(
                    "attached_lakehouse_workspace_name"):
                lh["attached_lakehouse_workspace_name"] = _ws_map.get(
                    (lh["attached_lakehouse_workspace_id"] or "").lower())
            findings = _scan_bytes(body_bytes, label, _SCAN_MD_BC)
        except Exception as e:
            out_rows.append(_empty_row(wid, wname, iid, name, now,
                                       f"scan error: {type(e).__name__}: {e}"))
            continue
        if not findings:
            out_rows.append(_empty_row(
                wid, wname, iid, name, now,
                lh_id=lh.get("attached_lakehouse_id"),
                lh_name=lh.get("attached_lakehouse_name"),
                lh_wsid=lh.get("attached_lakehouse_workspace_id"),
                lh_wsname=lh.get("attached_lakehouse_workspace_name"),
            ))
            continue
        for f in findings:
            # API mode: source_lakehouse_* stays NULL — the notebook lives in
            # a workspace, not a single lakehouse, so there's no source LH.
            out_rows.append(_row_from_finding(ns, f, wid, wname, iid, name, now))
    return iter(out_rows)


# --- Lakehouse-mode partition: bytes already loaded by binaryFile reader -----
def scan_partition_lakehouse(rows):
    from datetime import datetime, timezone
    rows = list(rows)
    if not rows:
        return iter([])
    ns = _build_scanner_ns()
    _scan_bytes = ns["scan_notebook_bytes"]
    _extract_lh = ns["_extract_attached_lakehouse"]
    _ws_map = _WS_NAME_BY_ID_BC or {}
    now = datetime.now(timezone.utc)
    out_rows = []
    # Source-LH workspace + LH IDs are constant for every row in this mode —
    # every file lives in the same mounted/configured lakehouse. Auto-filled
    # by cell 2 from notebookutils.runtime.context when the user left them
    # blank, so output rows now show real GUIDs instead of "lakehouse".
    wid = _LH_WS_BC
    wname = _LH_WSN_BC
    lh_src_id = _LH_ID_BC or None
    lh_src_name = _LH_LHN_BC or None
    for r in rows:
        try:
            file_path = r["path"] if "path" in r.__fields__ else r.path
        except Exception:
            file_path = getattr(r, "path", "") or ""
        try:
            body = r["content"] if "content" in r.__fields__ else r.content
        except Exception:
            body = getattr(r, "content", b"") or b""
        nb_id = file_path
        nb_name = file_path.rsplit("/", 1)[-1]
        try:
            lh = _extract_lh(body or b"", file_path)
            if lh.get("attached_lakehouse_workspace_id") and not lh.get(
                    "attached_lakehouse_workspace_name"):
                lh["attached_lakehouse_workspace_name"] = _ws_map.get(
                    (lh["attached_lakehouse_workspace_id"] or "").lower())
            findings = _scan_bytes(body or b"", file_path, _SCAN_MD_BC)
        except Exception as e:
            out_rows.append(_empty_row(wid, wname, nb_id, nb_name, now,
                                       f"scan error: {type(e).__name__}: {e}",
                                       lh_src_id=lh_src_id,
                                       lh_src_name=lh_src_name))
            continue
        if not findings:
            out_rows.append(_empty_row(
                wid, wname, nb_id, nb_name, now,
                lh_id=lh.get("attached_lakehouse_id"),
                lh_name=lh.get("attached_lakehouse_name"),
                lh_wsid=lh.get("attached_lakehouse_workspace_id"),
                lh_wsname=lh.get("attached_lakehouse_workspace_name"),
                lh_src_id=lh_src_id,
                lh_src_name=lh_src_name,
            ))
            continue
        for f in findings:
            out_rows.append(_row_from_finding(ns, f, wid, wname, nb_id, nb_name,
                                              now, lh_src_id=lh_src_id,
                                              lh_src_name=lh_src_name))
    return iter(out_rows)


# --- Result schema (unified for v2) -----------------------------------------
result_schema = StructType([
    StructField("workspace_id",                      StringType(), True),
    StructField("workspace_name",                    StringType(), True),
    StructField("source_lakehouse_id",               StringType(), True),
    StructField("source_lakehouse_name",             StringType(), True),
    StructField("notebook_id",                       StringType(), True),
    StructField("display_name",                      StringType(), True),
    StructField("attached_lakehouse_id",             StringType(), True),
    StructField("attached_lakehouse_name",           StringType(), True),
    StructField("attached_lakehouse_workspace_id",   StringType(), True),
    StructField("attached_lakehouse_workspace_name", StringType(), True),
    StructField("part_path",                         StringType(), True),
    StructField("source_kind",                       StringType(), True),
    StructField("cell_index",                        IntegerType(), True),
    StructField("line_number",                       IntegerType(), True),
    StructField("finding_type",                      StringType(), True),
    StructField("category",                          StringType(), True),
    StructField("severity",                          StringType(), True),
    StructField("message",                           StringType(), True),
    StructField("code_snippet",                      StringType(), True),
    StructField("url",                               StringType(), True),
    StructField("direction",                         StringType(), True),
    StructField("dest_workspace_id",                 StringType(), True),
    StructField("dest_workspace_name",               StringType(), True),
    StructField("cross_workspace",                   BooleanType(), True),
    StructField("error",                             StringType(), True),
    StructField("scanned_at",                        TimestampType(), True),
])


if SOURCE_MODE == "lakehouse":
    findings_rdd = inventory_df.rdd.mapPartitions(scan_partition_lakehouse)
else:
    findings_rdd = inventory_df.rdd.mapPartitions(fetch_partition)

findings_df = spark.createDataFrame(findings_rdd, schema=result_schema).cache()
print(f"Findings rows: {findings_df.count():,}")


In [ ]:
# === Persist findings + summary displays =====================================
from pyspark.sql import functions as F

if WRITE_TO_DEFAULT_LAKEHOUSE:
    findings_df.write.mode("overwrite").option(
        "overwriteSchema", "true").saveAsTable(output_target)
else:
    (findings_df.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .save(output_target))

print(f"Findings persisted to: {output_target}")

# Register a temp view for SQL displays
findings_df.createOrReplaceTempView("findings_v")

# 1. Severity rollup ---------------------------------------------------------
print("\n=== Findings by severity ===")
display(spark.sql(
    "SELECT COALESCE(severity, '(no findings)') AS severity, "
    "       COUNT(*) AS n "
    "FROM findings_v "
    "GROUP BY severity "
    "ORDER BY CASE severity "
    "  WHEN 'critical' THEN 0 WHEN 'high' THEN 1 "
    "  WHEN 'medium' THEN 2 WHEN 'low' THEN 3 ELSE 4 END"
))

# 2. Category x severity rollup ---------------------------------------------
print("\n=== Findings by category x severity ===")
display(spark.sql(
    "SELECT category, severity, COUNT(*) AS n "
    "FROM findings_v "
    "WHERE finding_type IS NOT NULL "
    "GROUP BY category, severity "
    "ORDER BY category, severity"
))

# 3. Top notebooks by critical+high count -----------------------------------
print("\n=== Top notebooks (critical + high count) ===")
display(spark.sql(
    "SELECT workspace_name, display_name, notebook_id, "
    "       SUM(CASE WHEN severity='critical' THEN 1 ELSE 0 END) AS critical_n, "
    "       SUM(CASE WHEN severity='high'     THEN 1 ELSE 0 END) AS high_n, "
    "       SUM(CASE WHEN severity='medium'   THEN 1 ELSE 0 END) AS medium_n, "
    "       SUM(CASE WHEN severity='low'      THEN 1 ELSE 0 END) AS low_n "
    "FROM findings_v "
    "WHERE finding_type IS NOT NULL "
    "GROUP BY workspace_name, display_name, notebook_id "
    "HAVING critical_n + high_n > 0 "
    "ORDER BY critical_n DESC, high_n DESC LIMIT 50"
))

# 4. URL direction breakdown ------------------------------------------------
print("\n=== URL findings by direction ===")
display(spark.sql(
    "SELECT direction, COUNT(*) AS n "
    "FROM findings_v "
    "WHERE finding_type = 'url' "
    "GROUP BY direction ORDER BY n DESC"
))

# 5. Cross-workspace data flows ---------------------------------------------
print("\n=== Cross-workspace flows ===")
display(spark.sql(
    "SELECT workspace_name AS src_ws, dest_workspace_name AS dest_ws, "
    "       direction, COUNT(*) AS n "
    "FROM findings_v "
    "WHERE cross_workspace = true "
    "GROUP BY workspace_name, dest_workspace_name, direction "
    "ORDER BY n DESC LIMIT 100"
))

# 6. Notebooks grouped by attached lakehouse --------------------------------
print("\n=== Notebooks by attached lakehouse ===")
display(spark.sql(
    "SELECT attached_lakehouse_workspace_name AS lh_ws, "
    "       attached_lakehouse_name AS lakehouse, "
    "       attached_lakehouse_id, "
    "       COUNT(DISTINCT notebook_id) AS notebooks, "
    "       COUNT(*) AS total_findings "
    "FROM findings_v "
    "WHERE attached_lakehouse_id IS NOT NULL "
    "GROUP BY attached_lakehouse_workspace_name, attached_lakehouse_name, "
    "         attached_lakehouse_id "
    "ORDER BY notebooks DESC, total_findings DESC LIMIT 200"
))

# 7. Cross-workspace lakehouse attachments ----------------------------------
print("\n=== Notebooks whose attached lakehouse is in a different workspace ===")
display(spark.sql(
    "SELECT workspace_name AS notebook_ws, display_name AS notebook, "
    "       attached_lakehouse_workspace_name AS lakehouse_ws, "
    "       attached_lakehouse_name AS lakehouse, "
    "       COUNT(*) AS findings "
    "FROM findings_v "
    "WHERE attached_lakehouse_workspace_id IS NOT NULL "
    "  AND workspace_id IS NOT NULL "
    "  AND LOWER(attached_lakehouse_workspace_id) <> LOWER(workspace_id) "
    "GROUP BY workspace_name, display_name, "
    "         attached_lakehouse_workspace_name, attached_lakehouse_name "
    "ORDER BY findings DESC LIMIT 200"
))

# 8. Notebooks flagged as needing migration / update (fastcp etc.) ----------
print("\n=== Notebooks needing update / migration ===")
display(spark.sql(
    "SELECT workspace_name, display_name, notebook_id, "
    "       attached_lakehouse_name, "
    "       COUNT(*) AS needs_update_findings, "
    "       MIN(cell_index) AS first_cell, "
    "       MIN(line_number) AS first_line, "
    "       FIRST(message) AS sample_message, "
    "       FIRST(code_snippet) AS sample_snippet "
    "FROM findings_v "
    "WHERE category = 'needs_update' "
    "GROUP BY workspace_name, display_name, notebook_id, attached_lakehouse_name "
    "ORDER BY needs_update_findings DESC, workspace_name, display_name LIMIT 500"
))

# 9. Notebook review queue --------------------------------------------------
# Tag each notebook for human triage. Priority order:
#   critical_secret  -> any credential_access / critical severity finding
#   cross_workspace_write -> writes to a different workspace
#   dynamic_exec     -> eval/exec/decoded execution
#   high_entropy     -> obfuscated payloads
#   potential_write  -> write API used but no literal URL/workspace tag
#   needs_update     -> Fabric API migration required (fastcp etc.)
#   url_writes_only  -> writes to literal external URLs only
#   informational    -> only reference URLs / low severity
print("\n=== Notebook review queue ===")
display(spark.sql("""
    SELECT
      workspace_name, display_name, notebook_id,
      MAX(CASE WHEN severity='critical' THEN 1 ELSE 0 END)        AS has_critical,
      MAX(CASE WHEN severity='high'     THEN 1 ELSE 0 END)        AS has_high,
      MAX(CASE WHEN cross_workspace = true THEN 1 ELSE 0 END)      AS has_cross_ws,
      MAX(CASE WHEN finding_type='potential_write' THEN 1 ELSE 0 END) AS has_potential_write,
      MAX(CASE WHEN finding_type='dynamic_exec' THEN 1 ELSE 0 END)    AS has_dynamic_exec,
      MAX(CASE WHEN finding_type='entropy' THEN 1 ELSE 0 END)         AS has_entropy,
      MAX(CASE WHEN finding_type='url' AND direction='write' THEN 1 ELSE 0 END) AS has_url_write,
      MAX(CASE WHEN category='needs_update' THEN 1 ELSE 0 END)        AS has_needs_update,
      COUNT(*) AS total_findings,
      CASE
        WHEN MAX(CASE WHEN severity='critical' THEN 1 ELSE 0 END) = 1
          THEN 'critical_review'
        WHEN MAX(CASE WHEN cross_workspace = true THEN 1 ELSE 0 END) = 1
          THEN 'cross_workspace_write'
        WHEN MAX(CASE WHEN finding_type='dynamic_exec' THEN 1 ELSE 0 END) = 1
          THEN 'dynamic_exec_review'
        WHEN MAX(CASE WHEN finding_type='entropy' THEN 1 ELSE 0 END) = 1
          THEN 'high_entropy_review'
        WHEN MAX(CASE WHEN finding_type='potential_write' THEN 1 ELSE 0 END) = 1
             AND MAX(CASE WHEN finding_type='url' AND direction='write' THEN 1 ELSE 0 END) = 0
          THEN 'flag_for_review'
        WHEN MAX(CASE WHEN category='needs_update' THEN 1 ELSE 0 END) = 1
          THEN 'needs_update_review'
        WHEN MAX(CASE WHEN finding_type='url' AND direction='write' THEN 1 ELSE 0 END) = 1
          THEN 'url_writes_only'
        WHEN MAX(CASE WHEN finding_type IS NOT NULL THEN 1 ELSE 0 END) = 1
          THEN 'informational'
        ELSE 'clean'
      END AS status
    FROM findings_v
    GROUP BY workspace_name, display_name, notebook_id
    ORDER BY has_critical DESC, has_high DESC, has_cross_ws DESC,
             has_needs_update DESC, total_findings DESC
"""))

# 10. Sample of credential / critical findings (redacted snippets) ----------
print("\n=== Sample critical/high findings ===")
display(spark.sql(
    "SELECT workspace_name, display_name, cell_index, line_number, "
    "       finding_type, category, severity, message, code_snippet "
    "FROM findings_v "
    "WHERE severity IN ('critical', 'high') "
    "ORDER BY severity, workspace_name, display_name "
    "LIMIT 200"
))
